# Paper figures

Every main-text figure of *Paleoclimate Boundary Conditions as an Out-of-Sample Test for
the Forced Response of Ocean Climate Emulators*, plus the baseline comparisons from the
supplement, in one notebook. Setup and shared helpers live in the first two cells; each
figure is then a single self-contained cell that uses them.

| cell | paper figure | output |
|---|---|---|
| 1 | Fig. 1 | `Holocene_PiControl_Comparison.png` |
| 2 | Fig. 2 | `Change_in_Pacific_SST_Seasonal_Cycle_thetao.png` |
| 3 | Fig. 3 | `Pacific_Seasonal_Change_thetao.png` |
| 4 | Fig. 4 | `Variability_Plot_Depth.png` |
| 5 | Fig. 5 (and Fig. S27) | `Variability_Change_0-200m.png`, `Variability_Change_200-1000m.png` |
| 6 | Fig. 6 | `Responses_Full_Comparisons_Metrics_thetao.png` |
| 7 | Fig. 7 | `Responses_Full_Maps_Multi_Depth_Comparisons_thetao.png` |
| 8 | Fig. 8 | `Responses_Specialist_Comparisons_Updated_thetao_Southern.png` |
| 9 | Fig. 9 | `Residual_Analysis_thetao_Southern.png` |
| 10 | Fig. 10 | `MultiExperiment_MultiBasin_Epoch_Comparisons.png` |
| 11–13 | Figs. S20–S22 | baseline (LRO / forcing-only) comparisons |

**Inputs.** Two kinds of store, both produced by the scripts in this repository:

* *rollouts* — `Notebooks/Generate_Rollouts_Recon.py` writes
  `BVP_<NAME_MODIFIER>_<climate>_IC_Rollout_Epoch_<N>.zarr` (100-yr monthly rollout) and
  `BVP_CESM2_<NAME_MODIFIER>_<base>_Epoch_<N>_Response_<target>_BC[_<forcing>...]/yearly_means.zarr`
  (5-member response ensembles, annual means on `year` × `init_time`).
* *truth* — the processed CESM2 stores written by `Preprocessing/`.

The emulator weights are in `Weights/`; see that folder's README for which checkpoint
backs which rollout.

**Sign convention.** $\mathcal{F}_{\mathrm{pi}}$ is perturbed piControl → midHolocene and
$\mathcal{F}_{\mathrm{mH}}$ midHolocene → piControl, so the two are compared under
$R(\mathcal{F}_{\mathrm{pi}}) \leftrightarrow -R(\mathcal{F}_{\mathrm{mH}})$: wherever a
midHolocene-trained response is scored, the truth is negated first.

In [ ]:
import calendar
import os
import string
import warnings

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.lines import Line2D
from scipy import stats
from xarrayutils.plotting import linear_piecewise_scale

warnings.filterwarnings("ignore")

VAR = "thetao"
UNITS = "°C" if VAR == "thetao" else "PSU"
VAR_SYMBOL = r"$\theta_O$" if VAR == "thetao" else "$s$"

BASE_PATH_EMU = "/pscratch/sd/a/asubel/Chapter_2/Preds"
BASE_PATH_TRUTH = "/pscratch/sd/a/asubel/Data/Chapter_2"
PATH_GRID = f"{BASE_PATH_TRUTH}/CESM2_piControl.zarr"
PATH_MASKS = f"{BASE_PATH_TRUTH}/Basin_Mask.nc"

FIG_DIR = "./Figures_For_Paper_Final"
os.makedirs(FIG_DIR, exist_ok=True)

# CESM2 references.  The detrended stores are the ones every response is scored against.
TRUTH = {
    "piControl": f"{BASE_PATH_TRUTH}/CESM2_piControl_Detrended.zarr",
    "Holocene": f"{BASE_PATH_TRUTH}/Holocene_Processed_Detrended.zarr",
    "1pct": f"{BASE_PATH_TRUTH}/CESM2_1pctCO2_remade.zarr",
}

# 100-yr monthly rollouts, one per (emulator, boundary-forcing climate) pair.
ROLLOUT = {
    ("pi", "piControl"): f"{BASE_PATH_EMU}/CM4_Reconstruct_Baseline_CESM2_piControl.zarr",
    ("pi", "Holocene"): f"{BASE_PATH_EMU}/CM4_Reconstruct_Baseline_CESM2_Holocene.zarr",
    ("mH", "Holocene"): f"{BASE_PATH_EMU}/CM4_Reconstruct_Holocene_CESM2_Holocene.zarr",
    ("mH", "piControl"): f"{BASE_PATH_EMU}/CM4_Reconstruct_Holocene_CESM2_piControl.zarr",
}

# Response ensembles.  Both emulators are reported at a checkpoint from the early,
# pre-degradation regime of Section 5.2 rather than at the lowest test RMSE.  Epoch 65 of
# the midHolocene emulator is the checkpoint the paper excludes: it is unstable in the
# North Atlantic and its response magnitudes there are nonphysical, which also blows up the
# colour scale of the Atlantic row in Figure 6.
RESPONSE_PREFIX = {"pi": "BVP_CESM2_Reconstructed_Baseline", "mH": "BVP_CESM2_Reconstructed_Holocene"}
EPOCH_PI = 50
EPOCH_MH = 70
FULL_FORCING = ["hfds", "tauuo", "tauvo", "orb"]

# Single-forcing emulators (Fig. 8), each at its own epoch.
SPECIALIST = {
    "hfds": f"{BASE_PATH_EMU}/BVP_CESM2_Reconstructed_HFDS_piControl_Epoch_70_Response_Holocene_BC",
    "tauuo": f"{BASE_PATH_EMU}/BVP_CESM2_Reconstructed_TAUUO_piControl_Epoch_41_Response_Holocene_BC",
    "tauvo": f"{BASE_PATH_EMU}/BVP_CESM2_Reconstructed_TAUVO_piControl_Epoch_38_Response_Holocene_BC",
}

# Baselines (Section S2): the local linear regression operator, one store per anomaly mode,
# and the boundary-forcing-only network.
LRO_LAMBDA, LRO_SCALING = 0.1, "per_sample"
LRO_MEAN = f"{BASE_PATH_EMU}/LRO_annual_lam{LRO_LAMBDA:g}_{LRO_SCALING}"
LRO_SEASONAL = f"{BASE_PATH_EMU}/LRO_monthly_seasonal_lam{LRO_LAMBDA:g}_{LRO_SCALING}"
LRO_DESEASON = f"{BASE_PATH_EMU}/LRO_monthly_deseason_lam{LRO_LAMBDA:g}_{LRO_SCALING}"
BFO = (f"{BASE_PATH_EMU}/BFO_CESM2_BoundaryOnly_piControl_Epoch_final"
       f"_Response_Holocene_BC_hfds_tauuo_tauvo_orb")

MAX_DEPTH = 1000            # every response figure is scored over the upper 1000 m
BASINS = ["Pacific", "Atlantic", "Southern", "Indian"]

# Boxes quoted in the text, drawn on the section figures.
METRIC_REGIONS = [
    {"name": "North Lats", "basin": "Pacific", "lat": (20, 90), "depth": (0, 1000), "color": "#D55E00"},
    {"name": "Tropics", "basin": "Pacific", "lat": (-20, 20), "depth": (0, 1000), "color": "#0072B2"},
    {"name": "Surface", "basin": "Atlantic", "lat": (-35, 65), "depth": (0, 150), "color": "#0072B2"},
    {"name": "Tropics", "basin": "Indian", "lat": (-20, 20), "depth": (0, 1000), "color": "#0072B2"},
    {"name": "Westerlies Shift", "basin": "Southern", "lat": (-52, -35), "depth": (0, 1000), "color": "#009E73"},
    {"name": "Surface Heating", "basin": "Southern", "lat": (-70, -52), "depth": (0, 200), "color": "#D55E00"},
]

PROJECTION = ccrs.PlateCarree(central_longitude=180)
RNG_SEED = 0               # the Fig. 1 histograms subsample time steps

## Shared helpers

Grid metrics, the zonal-mean response, the area-weighted scores, and the significance
stippling. Everything below this cell uses these rather than redefining them.

In [ ]:
def get_grid_metrics_and_masks(grid_ref_path=PATH_GRID, mask_path=PATH_MASKS):
    """dx, dz, areacello and the four basin masks.

    `Basin_Mask.nc` is y=154 while the model fields are y=180, so multiplying by a basin
    mask is also what restricts a panel to the emulator's domain (y <= 64).
    """
    grid_ref = xr.open_zarr(grid_ref_path, consolidated=True)
    radius = 6.378e6
    lon_b_y_center = (grid_ref.lon_b[:-1].values + grid_ref.lon_b[1:].values) / 2
    dx = xr.DataArray(
        lon_b_y_center[:, 1:] - lon_b_y_center[:, :-1],
        dims=("y", "x"),
        coords={"y": grid_ref.y, "x": grid_ref.x},
    )
    dx = (dx / 360) * 2 * np.pi * np.cos((grid_ref.lat.values / 360) * 2 * np.pi) * radius

    mask_data = xr.open_dataset(mask_path)
    mask_data = xr.where(mask_data == 4, 5, mask_data)          # merge 4 into the Indian
    masks = {
        "Atlantic": xr.where(mask_data.basin == 2, 1.0, np.nan),
        "Pacific": xr.where(mask_data.basin == 3, 1.0, np.nan),
        "Indian": xr.where(mask_data.basin == 5, 1.0, np.nan),
        "Southern": xr.where(mask_data.basin == 1, 1.0, np.nan),
    }
    return dx, grid_ref["dz"], masks, grid_ref["areacello"]


DX, DZ, MASKS, AREACELLO = get_grid_metrics_and_masks()


def basin_mask(name):
    """Basin mask, with the Southern Ocean cut at 35°S so it does not overlap the others."""
    mask = MASKS[name].copy()
    if name == "Southern":
        mask.loc[{"y": slice(-35, None)}] = np.nan
    return mask


def basin_bounds(mask):
    lat = mask * mask.lat
    return int(lat.min().item()) - 2, int(lat.max().item()) + 2


def response_path(emulator, epoch, forcings=None, target=None):
    """Path to one response ensemble.

    `forcings=None` is the control member of the pair — the emulator run on its own
    training climate's forcing — which every response is differenced against.
    """
    base = "piControl" if emulator == "pi" else "Holocene"
    target = target or ("Holocene" if emulator == "pi" else "piControl")
    folder = f"{RESPONSE_PREFIX[emulator]}_{base}_Epoch_{epoch}_Response_{target}_BC"
    if forcings:
        folder += "".join(f"_{f}" for f in forcings)
    return f"{BASE_PATH_EMU}/{folder}/yearly_means.zarr"


def zonal_profile(field, mask, wet):
    """dx- and basin-weighted zonal mean of a (..., lev, y, x) field."""
    return (field * DX * mask).sum("x", skipna=True) / (DX * mask * wet).sum("x", skipna=True)


def wet_mask(ds, var=VAR, time_dim="time"):
    """The 1/NaN ocean mask of a store, taken from its first time step."""
    wet = (ds[var] * 0 + 1).isel({time_dim: 0}, drop=True)
    if "init_time" in wet.dims:
        wet = wet.isel(init_time=0, drop=True)
    return wet


def zonal_mean_response(ctrl, pert, mask, var=VAR, max_depth=MAX_DEPTH,
                        window=slice(-25, None), is_truth=False, collapse=True):
    """Zonal-mean response, time-averaged over `window`.

    The truth stores carry a monthly `time` axis and are averaged over their last 100
    years; the emulator response stores carry annual means on `year` plus an `init_time`
    ensemble.  With `collapse=False` the ensemble dimension is kept, which is what the
    stippling needs.
    """
    time_dim = "time" if is_truth else "year"
    resp = (pert[var] - ctrl[var]).sel(lev=slice(0, max_depth))
    zonal = zonal_profile(resp, mask, wet_mask(ctrl, var, time_dim))
    sel = slice(-100 * 12, None) if is_truth else window
    out = zonal.isel({time_dim: sel}).mean(time_dim)
    if collapse and "init_time" in out.dims:
        out = out.mean("init_time")
    lo, hi = basin_bounds(mask)
    return out.sel(y=slice(lo, hi)).compute()


def ensemble_stats(field, dim="init_time"):
    """(mean, spread, n) over the ensemble dimension, tolerating a single member."""
    if dim in field.dims:
        return field.mean(dim), field.std(dim), field.sizes[dim]
    return field, xr.zeros_like(field), 1


def truth_chunk_means(ctrl, pert, var=VAR, chunk_years=10, max_depth=None):
    """The CESM2 pseudo-ensemble: non-overlapping `chunk_years`-long means of the response.

    This is the internal-variability estimate the significance test uses, in place of an
    ensemble the single CESM2 pair does not have.
    """
    resp = pert[var] - ctrl[var]
    if max_depth is not None:
        resp = resp.sel(lev=slice(0, max_depth))
    chunk_size = chunk_years * 12
    n_chunks = resp.sizes["time"] // chunk_size
    resp = resp.isel(time=slice(-n_chunks * chunk_size, None))
    chunks = (resp.coarsen(time=chunk_size, boundary="trim")
              .construct({"time": ("chunk", "month_in_chunk")}).mean("month_in_chunk"))
    return chunks, n_chunks


def depth_mean(field, band):
    """dz-weighted vertical average over a depth band."""
    dz_band = DZ.sel(lev=slice(*band)).fillna(0.0)
    return field.sel(lev=slice(*band)).weighted(dz_band).mean("lev", skipna=True)


def section_area(max_depth=MAX_DEPTH):
    """Weights for a depth-latitude section: cell thickness times the zonal-mean width."""
    return (DZ * DX.mean("x")).sel(lev=slice(0, max_depth))


def corr(a, b, area):
    """Uncentred (pattern) correlation, area-weighted."""
    return float((area * a * b).sum() / np.sqrt((area * a ** 2).sum() * (area * b ** 2).sum()))


def rmse(a, b, area):
    return float(np.sqrt((area * (a - b) ** 2).sum() / area.sum()))


def wstd(a, area):
    w = area / area.sum()
    return float(np.sqrt((w * (a - (w * a).sum()) ** 2).sum()))


def amp_ratio(model, truth, area):
    """sigma_model / sigma_truth: 1 when the response amplitude matches, < 1 when damped."""
    return wstd(model, area) / wstd(truth, area)


def score(model, truth, area):
    return {"rho": corr(model, truth, area), "rmse": rmse(model, truth, area),
            "amp": amp_ratio(model, truth, area)}


def insignificance_mask(response, spread, n, ci=0.95, use_sem=True):
    """True where the two-sided CI on the mean response includes zero; None for n < 2."""
    if n is None or n < 2:
        return None
    t_crit = stats.t.ppf(1.0 - (1.0 - ci) / 2.0, df=n - 1)
    multiplier = t_crit / np.sqrt(n) if use_sem else t_crit
    return np.abs(response) <= multiplier * spread


HATCH = dict(levels=[0.5, 1.5], colors="none", hatches=["....."], zorder=4)


def stipple_section(ax, mask):
    if mask is None or not bool(mask.any()):
        return
    xr.where(mask, 1, np.nan).plot.contourf(
        ax=ax, add_colorbar=False, yincrease=False, add_labels=False, **HATCH)


def stipple_map(ax, lon, lat, mask):
    if mask is None or not bool(mask.any()):
        return
    ax.contourf(lon, lat, mask, transform=ccrs.PlateCarree(), **HATCH)


def seasonal_range_profile(path, mask, var=VAR, max_depth=500, months=600):
    """Zonal-mean depth profile of the SON minus MAM range over the last `months` months."""
    field = xr.open_zarr(path, consolidated=True)[var].sel(lev=slice(0, max_depth))
    seasons = field.isel(time=slice(-months, None)).groupby("time.season").mean("time").compute()
    rng = seasons.sel(season="SON") - seasons.sel(season="MAM")
    profile = zonal_profile(rng, mask, rng.notnull() * 1.0)
    lo, hi = basin_bounds(mask)
    return profile.sel(y=slice(lo, hi)).compute()


def deseasonalize(da):
    """Remove the monthly climatology, leaving interannual variability."""
    return da.groupby("time.month") - da.groupby("time.month").mean("time")


def month_key(times):
    """Monotonic year*12 + month index.

    The stores disagree on the day within the month (rollouts stamp the 14th, the truth the
    15th) and do not all share a cftime calendar class, either of which breaks a direct
    `sel(time=slice(...))`.  Year and month are unambiguous across all of them.
    """
    t = xr.DataArray(times, dims="time")
    return (t.dt.year * 12 + t.dt.month).values.astype(np.int64)


def panel_letter(i):
    return r"$\mathbf{" + string.ascii_uppercase[i] + r"}$)"


# Column headings shared by the two time-mean response figures.
RESPONSE_COLUMN_TITLES = [
    "(midHolocene - piControl)",
    r"$\mathcal{E}(\mathcal{F}_{pi},\Phi_{\mathcal{F}_{pi}}^{[0]};\overline{\boldsymbol{\tau}}_{mH},I_{mH}) - "
    r"\mathcal{E}(\mathcal{F}_{pi},\Phi_{\mathcal{F}_{pi}}^{[0]};\overline{\boldsymbol{\tau}}_{pi},I_{pi})$",
    r"$\mathcal{E}(\mathcal{F}_{mH},\Phi_{\mathcal{F}_{mH}}^{[0]};\overline{\boldsymbol{\tau}}_{pi},I_{pi}) - "
    r"\mathcal{E}(\mathcal{F}_{mH},\Phi_{\mathcal{F}_{mH}}^{[0]};\overline{\boldsymbol{\tau}}_{mH},I_{mH})$",
]


print(f"grid {dict(xr.open_zarr(TRUTH['piControl']).sizes)}")

## Figure 1 — piControl against midHolocene

What changes between the two CESM2 experiments: the time-mean boundary forcings (A–C),
the upper-1000 m distributions of temperature and salinity with a 1% CO$_2$ run for
scale (D–E), and the seasonal cycle of the upper 200 m (F).

In [ ]:
MAX_DEPTH_HIST = 1000
MAX_DEPTH_SEASONAL = 200
N_TIME_SAMPLES = 40          # random months per store for the distributions
INCLUDE_EMULATOR_HIST = False

HIST_SOURCES = {
    "CESM2 piControl": TRUTH["piControl"],
    "CESM2 Holocene": TRUTH["Holocene"],
    "CESM2 1pct": TRUTH["1pct"],
    "Emulator piControl": ROLLOUT[("pi", "piControl")],
    "Emulator Holocene": ROLLOUT[("mH", "Holocene")],
}

STYLE = {
    "CESM2 1pct": {"color": "#FF8A8A", "linestyle": "-", "lw": 3.5},
    "CESM2 piControl": {"color": "black", "linestyle": "-", "lw": 3},
    "CESM2 Holocene": {"color": "#0072B2", "linestyle": "-", "lw": 3.5},
    "Emulator piControl": {"color": "gray", "linestyle": "--", "lw": 3},
    "Emulator Holocene": {"color": "#003366", "linestyle": "--", "lw": 3.5},
}


def forcing_difference(var):
    """midHolocene minus piControl time-mean forcing, over the midHolocene record."""
    ctrl = xr.open_zarr(TRUTH["piControl"])[var]
    holo = xr.open_zarr(TRUTH["Holocene"])[var]
    ctrl = ctrl.sel(time=slice(holo["time"][0].values, holo["time"][-1].values))
    return (holo.mean("time") - ctrl.mean("time")).compute()


def distribution_samples(var, rng):
    """Upper-1000 m values from `N_TIME_SAMPLES` random months of each store."""
    frames = []
    for name, path in HIST_SOURCES.items():
        ds = xr.open_zarr(path, consolidated=False).sel(y=slice(-90, 64))
        if name == "CESM2 1pct":
            ds = ds.sel(time=slice("0050-01-01", "0090-12-31"))   # the CO2-doubling window
        n = ds.sizes["time"]
        idx = (np.arange(n) if n < N_TIME_SAMPLES
               else rng.choice(n, N_TIME_SAMPLES, replace=False))
        sub = ds[var].isel(time=idx).sel(lev=slice(0, MAX_DEPTH_HIST))
        if var == "so":
            sub = xr.where((sub > 38) | (sub < 31), np.nan, sub)
        values = sub.values.flatten()
        source, experiment = name.split(" ")
        frames.append(pd.DataFrame({var: values[~np.isnan(values)],
                                    "Source": source, "Experiment": experiment}))
    return pd.concat(frames, ignore_index=True)


def seasonal_cycle(path, var=VAR):
    """Volume-weighted mean over the upper 200 m, as a monthly anomaly from the annual mean."""
    field = xr.open_zarr(path, consolidated=True)[var].sel(lev=slice(0, MAX_DEPTH_SEASONAL))
    weights = (AREACELLO * DZ.sel(lev=slice(0, MAX_DEPTH_SEASONAL))).fillna(0)
    cycle = field.weighted(weights).mean(["x", "y", "lev"]).groupby("time.month").mean("time")
    return (cycle - cycle.mean("month")).compute()


rng = np.random.default_rng(RNG_SEED)
forcing_maps = {v: forcing_difference(v) for v in ["hfds", "tauuo", "tauvo"]}
hist_temp = distribution_samples("thetao", rng)
hist_sal = distribution_samples("so", rng)
cycles = {name: seasonal_cycle(path) for name, path in HIST_SOURCES.items()
          if name != "CESM2 1pct"}

plt.style.use("seaborn-v0_8-talk")
fig, axs = plt.subplot_mosaic(
    "AB\nCD\nEF", figsize=(14, 14),
    per_subplot_kw={k: {"projection": ccrs.PlateCarree()} for k in "ABC"},
    gridspec_kw={"height_ratios": [1, 1, 1]})

for key, (var, title) in zip("ABC", [("hfds", "Heat Flux"), ("tauuo", "Zonal Surface Stress"),
                                     ("tauvo", "Meridional Surface Stress")]):
    ax = axs[key]
    ax.add_feature(cfeature.LAND, zorder=1, edgecolor="black", facecolor="lightgray")
    ax.coastlines()
    data = forcing_maps[var]
    vmax = np.abs(data).quantile(0.99 if var == "hfds" else 0.96).item()
    mesh = data.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(), cmap=cmocean.cm.balance,
                                vmin=-vmax, vmax=vmax, add_colorbar=False)
    cbar = fig.colorbar(mesh, ax=ax, orientation="horizontal", shrink=0.8, pad=0.08)
    cbar.set_label("(Holocene - piControl)")
    units = r" $[W/m^2]$" if var == "hfds" else r" $[N/m^2]$"
    ax.set_title(f"{key}) Forcing: {title} " + units, loc="left", fontsize=16)

bins_t = np.linspace(hist_temp["thetao"].min(), hist_temp["thetao"].max(), 60)
bins_s = np.linspace(hist_sal["so"].min(), hist_sal["so"].max(), 60)

for i, (name, style) in enumerate(STYLE.items()):
    if not INCLUDE_EMULATOR_HIST and "Emulator" in name:
        continue
    source, experiment = name.split(" ")
    dt = hist_temp[(hist_temp.Source == source) & (hist_temp.Experiment == experiment)]["thetao"]
    ds = hist_sal[(hist_sal.Source == source) & (hist_sal.Experiment == experiment)]["so"]
    if source == "CESM2":
        kw = dict(density=True, alpha=0.8 if i == 1 else 0.5, color=style["color"],
                  label=name, histtype="stepfilled", edgecolor=style["color"], linewidth=3)
    else:
        kw = dict(density=True, color=style["color"], label=name,
                  histtype="step", linewidth=2.5, linestyle="--")
    axs["D"].hist(dt, bins=bins_t, **kw)
    axs["E"].hist(ds, bins=bins_s, **kw)

axs["D"].set_title("D) Temperature Distribution", loc="left", fontsize=16)
axs["D"].set_xlabel("Potential Temperature (°C)")
axs["E"].set_title("E) Salinity Distribution", loc="left", fontsize=16)
axs["E"].set_xlabel("Salinity (PSU)")
for key in "DE":
    axs[key].set_ylabel("Probability Density")
    axs[key].grid(True, linestyle=":", alpha=0.7)
axs["D"].legend(loc="upper right", fontsize=12, frameon=True)

for name, cycle in cycles.items():
    cycle.plot(ax=axs["F"], **STYLE[name], label=name)
axs["F"].set_title("F) Seasonal Cycle", loc="left", fontsize=16)
axs["F"].set_ylabel("Monthly Anomaly (°C)")
axs["F"].set_xticks(range(1, 13))
axs["F"].set_xticklabels(calendar.month_abbr[1:])
axs["F"].grid(True, linestyle=":")
axs["F"].legend(loc="lower left", fontsize=12, ncol=1)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.savefig(f"{FIG_DIR}/Holocene_PiControl_Comparison.png", dpi=300, bbox_inches="tight")
plt.show()
plt.style.use("default")

## Figure 2 — equatorial Pacific SST seasonal cycle

Monthly SST anomalies about the annual mean, averaged over 1.5°S–1.5°N, differenced
between the two climates. Panel B differences each emulator on its own training climate
and so carries no cross-climate generalization; C and D are the generalization tests.

In [ ]:
EQUATORIAL_BOUNDS = (-1.5, 1.5)
PACIFIC_LON_BOUNDS = (120, 280)
FS = {"panel_title": 14, "axis_label": 16, "tick": 14, "cbar_label": 16, "cbar_tick": 14}


def hovmoller_anomaly(path, var=VAR):
    """Month × longitude SST anomaly about the annual mean, last 50 years of the store."""
    data = xr.open_zarr(path, consolidated=True)
    time_dim = "year" if "year" in data.dims else "time"
    window = slice(-600, None) if time_dim == "year" else slice(-3600, None)
    sst = data[var].isel({time_dim: window}).isel(lev=0).sel(
        y=slice(*EQUATORIAL_BOUNDS), x=slice(*PACIFIC_LON_BOUNDS)).mean("y")
    clim = sst.groupby(f"{time_dim}.month").mean(time_dim).compute()
    return clim - clim.mean("month")


def unweighted_corr(a, b):
    a_flat, b_flat = a.values.flatten(), b.values.flatten()
    valid = np.isfinite(a_flat) & np.isfinite(b_flat)
    return np.corrcoef(a_flat[valid], b_flat[valid])[0, 1]


hov = {key: hovmoller_anomaly(path) for key, path in {
    "truth_pi": TRUTH["piControl"], "truth_ho": TRUTH["Holocene"],
    "pi_pi": ROLLOUT[("pi", "piControl")], "pi_ho": ROLLOUT[("pi", "Holocene")],
    "mH_ho": ROLLOUT[("mH", "Holocene")], "mH_pi": ROLLOUT[("mH", "piControl")],
}.items()}

diffs = {
    "truth_diff": hov["truth_ho"] - hov["truth_pi"],
    "emu_ic_diff": hov["mH_ho"] - hov["pi_pi"],
    "emu_pi_response": hov["pi_ho"] - hov["pi_pi"],
    "emu_holo_response": hov["mH_pi"] - hov["mH_ho"],   # piControl-forced, so sign-flipped
}
titles = {
    "truth_diff": r"$\bf{A}$) CESM2: midHolo $-$ piC",
    "emu_ic_diff": r"$\bf{B}$) $\Omega(\mathcal{F}_{mH},\tau_{mH}) - \Omega(\mathcal{F}_{pi},\tau_{pi})$",
    "emu_pi_response": r"$\bf{C}$) $\Omega(\mathcal{F}_{pi},\tau_{mH}) - \Omega(\mathcal{F}_{pi},\tau_{pi})$",
    "emu_holo_response": r"$\bf{D}$) $\Omega(\mathcal{F}_{mH},\tau_{mH}) - \Omega(\mathcal{F}_{mH},\tau_{pi})$",
}

metrics = {}
for key in ["emu_ic_diff", "emu_pi_response", "emu_holo_response"]:
    reference = -diffs["truth_diff"] if key == "emu_holo_response" else diffs["truth_diff"]
    metrics[key] = {"corr": unweighted_corr(diffs[key], reference),
                    "rmse": float(np.sqrt(np.nanmean((diffs[key] - reference) ** 2)))}
    print(f"  {key:<18} rho={metrics[key]['corr']:.2f}  RMSE={metrics[key]['rmse']:.3f} {UNITS}")

fig, axes = plt.subplots(2, 2, figsize=(11, 12), sharex=True, sharey=True,
                         constrained_layout=True)
vmax = max(np.abs(m).max().item() for m in diffs.values())
norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)

for ax, key in zip(axes.flat, diffs):
    data = -diffs[key] if key == "emu_holo_response" else diffs[key]
    mesh = data.drop_vars(["lev"], errors="ignore").plot.contourf(
        ax=ax, cmap=cmocean.cm.balance, norm=norm, add_colorbar=False,
        levels=21, x="x", extend="both")
    ax.set_title(titles[key], loc="left", fontsize=FS["panel_title"])
    if key != "truth_diff":
        ax.text(0.03, 0.96,
                f"$\\rho$ = {metrics[key]['corr']:.2f}\nRMSE = {metrics[key]['rmse']:.3f}",
                transform=ax.transAxes, fontsize=FS["tick"], va="top", ha="left",
                bbox=dict(facecolor="white", alpha=0.85, edgecolor="lightgray",
                          boxstyle="round,pad=0.4"))
    ax.set_xlabel("Longitude", fontsize=FS["axis_label"])
    ax.set_ylabel("Month", fontsize=FS["axis_label"])
    ax.set_yticks(np.arange(1, 13))
    ax.set_yticklabels(list("JFMAMJJASOND"))
    ax.tick_params(axis="both", labelsize=FS["tick"])
    ax.grid(True, linestyle="--", alpha=0.5)
axes[0, 1].set_ylabel("")
axes[1, 1].set_ylabel("")

cbar = fig.colorbar(mesh, ax=axes, orientation="vertical", shrink=0.7, pad=0.02)
cbar.set_label(f"Δ SST Anomaly ({UNITS})", fontsize=FS["cbar_label"])
cbar.ax.tick_params(labelsize=FS["cbar_tick"])
plt.savefig(f"{FIG_DIR}/Change_in_Pacific_SST_Seasonal_Cycle_{VAR}.png",
            dpi=300, bbox_inches="tight")
plt.show()

## Figure 3 — Pacific change in the seasonal range

The same seasonality test below the surface: Δ(SON − MAM) of potential temperature over
the upper 500 m, so skill here cannot come from the surface forcing alone. The 0.25 °C
contour is drawn on every panel to make the depth of the change comparable by eye.

In [ ]:
SEASONAL_MAX_DEPTH = 500
FS = {"suptitle": 24, "panel_title": 22, "panel_label": 18, "cbar_label": 18,
      "tick": 14, "cbar_tick": 16}

pacific = MASKS["Pacific"]
profiles = {key: seasonal_range_profile(path, pacific, max_depth=SEASONAL_MAX_DEPTH)
            for key, path in {
                "truth_pi": TRUTH["piControl"], "truth_ho": TRUTH["Holocene"],
                "pi_pi": ROLLOUT[("pi", "piControl")], "pi_ho": ROLLOUT[("pi", "Holocene")],
                "mH_ho": ROLLOUT[("mH", "Holocene")], "mH_pi": ROLLOUT[("mH", "piControl")],
            }.items()}

diffs = {
    "truth_diff": profiles["truth_ho"] - profiles["truth_pi"],
    "emu_ic_diff": profiles["mH_ho"] - profiles["pi_pi"],
    "emu_pi_response": profiles["pi_ho"] - profiles["pi_pi"],
    "emu_holo_response": profiles["mH_ho"] - profiles["mH_pi"],
}
titles = {
    "truth_diff": r"CESM2: midHolo $-$ piC",
    "emu_ic_diff": r"$\Omega(\mathcal{F}_{mH},\tau_{mH}) - \Omega(\mathcal{F}_{pi},\tau_{pi})$",
    "emu_pi_response": r"$\Omega(\mathcal{F}_{pi},\tau_{mH}) - \Omega(\mathcal{F}_{pi},\tau_{pi})$",
    "emu_holo_response": r"$\Omega(\mathcal{F}_{mH},\tau_{mH}) - \Omega(\mathcal{F}_{mH},\tau_{pi})$",
}

area = section_area(SEASONAL_MAX_DEPTH)
reference = diffs["truth_diff"]
metrics = {}
for key in ["emu_ic_diff", "emu_pi_response", "emu_holo_response"]:
    common = area.where(diffs[key].notnull() & reference.notnull())
    metrics[key] = score(diffs[key], reference, common)
    print(f"  {key:<18} rho={metrics[key]['rho']:.2f}  RMSE={metrics[key]['rmse']:.3f}  "
          f"amp={metrics[key]['amp']:.2f}")

fig, axes = plt.subplots(2, 2, figsize=(20, 12), constrained_layout=True)
vmax = max(np.abs(d).max().item() for d in diffs.values()) * 0.8
norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)

for i, (ax, key) in enumerate(zip(axes.flat, diffs)):
    mesh = diffs[key].plot.contourf(ax=ax, cmap=cmocean.cm.balance, norm=norm,
                                    add_colorbar=False, levels=25, yincrease=False)
    contour = diffs[key].plot.contour(ax=ax, colors="k", levels=[0.25], yincrease=False)
    ax.clabel(contour, inline=True, fontsize=14)
    ax.set_title(f"{panel_letter(i)} {titles[key]}", loc="left", fontsize=FS["panel_title"])
    ax.set_ylabel("Depth (m)", fontsize=FS["tick"])
    ax.set_xlabel("Latitude", fontsize=FS["tick"])
    ax.tick_params(labelsize=FS["tick"])
    ax.grid(True, linestyle=":", alpha=0.6)
    linear_piecewise_scale(500, 2, ax=ax)
    if key in metrics:
        ax.text(0.03, 0.04,
                f"$\\rho$ = {metrics[key]['rho']:.2f}\nRMSE = {metrics[key]['rmse']:.3f}",
                transform=ax.transAxes, fontsize=FS["panel_label"], va="bottom", ha="left",
                bbox=dict(facecolor="white", alpha=0.85, edgecolor="lightgray",
                          boxstyle="round,pad=0.4"))
axes[0, 0].set_xlabel("")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("")
axes[1, 1].set_ylabel("")

cbar = fig.colorbar(mesh, ax=axes, orientation="vertical", shrink=0.8, pad=0.02)
cbar.set_label(f"Δ(SON - MAM) ({UNITS})", fontsize=FS["cbar_label"])
cbar.ax.tick_params(labelsize=FS["cbar_tick"])
fig.suptitle(f"Pacific Change in Seasonal Range ({VAR})", fontsize=FS["suptitle"], y=1.03)
plt.savefig(f"{FIG_DIR}/Pacific_Seasonal_Change_{VAR}.png", dpi=300, bbox_inches="tight")
plt.show()

## Figure 4 — modes of variability

Niño3.4 and the Dipole Mode Index in both climates, their autocorrelations, the lag
between them, and the subsurface composite of the strongest 10% of Niño3.4 months. The
rollouts are forced with the full CESM2 boundary conditions here rather than the
climatology, since climatological forcing damps the variability being measured.

In [ ]:
INDEX_BOXES = {
    "Nino3.4": (190, 240, -5, 5),
    "DMI": {"West": (50, 70, -10, 10), "East": (90, 110, -10, 0)},
}
MAX_LAG, CHUNK_YEARS, LAST_N_MONTHS = 48, 20, 420

SERIES = {
    "Truth piControl": TRUTH["piControl"],
    "Truth Holocene": TRUTH["Holocene"],
    "Emu: piC -> piC": ROLLOUT[("pi", "piControl")],
    "Emu: piC -> Holo": ROLLOUT[("pi", "Holocene")],
    "Emu: Holo -> Holo": ROLLOUT[("mH", "Holocene")],
    "Emu: Holo -> piC": ROLLOUT[("mH", "piControl")],
}
SAFE = {"black": "#000000", "gray": "#999999", "blue": "#1E88E5",
        "orange": "#FFC107", "green": "#00584A", "purple": "#D81B60"}
STYLES = {
    "Truth piControl": {"c": SAFE["black"], "ls": "-", "lw": 2.5, "zorder": 0,
                        "label": r"CESM2 $\mathrm{pi}$"},
    "Truth Holocene": {"c": SAFE["gray"], "ls": "-", "lw": 2.5, "zorder": 0,
                       "label": r"CESM2 $\mathrm{mH}$"},
    "Emu: piC -> piC": {"c": SAFE["blue"], "ls": "-", "lw": 1.5, "zorder": 1,
                        "label": r"$\Omega(\mathcal{F}_{\mathrm{pi}}; \boldsymbol{\tau}_{\mathrm{pi}})$"},
    "Emu: piC -> Holo": {"c": SAFE["orange"], "ls": "--", "lw": 1.5, "zorder": 2,
                         "label": r"$\Omega(\mathcal{F}_{\mathrm{pi}}; \boldsymbol{\tau}_{\mathrm{mH}})$"},
    "Emu: Holo -> Holo": {"c": SAFE["green"], "ls": "-", "lw": 1.5, "zorder": 1,
                          "label": r"$\Omega(\mathcal{F}_{\mathrm{mH}}; \boldsymbol{\tau}_{\mathrm{mH}})$"},
    "Emu: Holo -> piC": {"c": SAFE["purple"], "ls": "--", "lw": 1.5, "zorder": 2,
                         "label": r"$\Omega(\mathcal{F}_{\mathrm{mH}}; \boldsymbol{\tau}_{\mathrm{pi}})$"},
}
PI_CONTEXT = ["Truth piControl", "Emu: piC -> piC", "Emu: Holo -> piC"]
HO_CONTEXT = ["Truth Holocene", "Emu: Holo -> Holo", "Emu: piC -> Holo"]


def area_mean(da):
    weights = AREACELLO.sel(y=da.y, x=da.x)
    return (da * weights).sum(dim=["y", "x"]) / weights.sum(dim=["y", "x"])


def compute_index(sst, boxes):
    """Area-mean anomaly in a box, or the difference of two boxes for the DMI."""
    anomaly = sst.groupby("time.month") - sst.groupby("time.month").mean("time")
    if isinstance(boxes, dict):
        parts = [area_mean(anomaly.sel(x=slice(*b[:2]), y=slice(*b[2:]))).compute()
                 for b in boxes.values()]
        return parts[0] - parts[1]
    return area_mean(anomaly.sel(x=slice(*boxes[:2]), y=slice(*boxes[2:]))).compute()


def autocorrelation(ts, max_lag=MAX_LAG, chunk_years=CHUNK_YEARS):
    """Mean and spread of the autocorrelation across `chunk_years`-long chunks."""
    vals = ts.values - ts.values.mean()
    per_chunk = chunk_years * 12
    n_chunks = max(len(vals) // per_chunk, 1)
    curves = []
    for i in range(n_chunks):
        chunk = vals[i * per_chunk:(i + 1) * per_chunk] if len(vals) >= per_chunk else vals
        curves.append([1.0] + [np.corrcoef(chunk[:-l], chunk[l:])[0, 1]
                               for l in range(1, max_lag + 1)])
    arr = np.array(curves)
    return np.arange(max_lag + 1), np.nanmean(arr, axis=0), np.nanstd(arr, axis=0)


def lagged_correlation(ts1, ts2, max_lag=MAX_LAG, chunk_years=CHUNK_YEARS):
    """Correlation of ts1 with ts2 lagged by up to `max_lag` months (ts1 leads)."""
    ts1, ts2 = xr.align(ts1, ts2, join="inner")
    v1, v2 = ts1.values, ts2.values
    per_chunk = chunk_years * 12
    n_chunks = len(v1) // per_chunk
    if n_chunks < 1:
        return None, None, None
    lags = np.arange(max_lag + 1)
    curves = []
    for i in range(n_chunks):
        a, b = v1[i * per_chunk:(i + 1) * per_chunk], v2[i * per_chunk:(i + 1) * per_chunk]
        curves.append([np.corrcoef(a, b)[0, 1] if l == 0
                       else np.corrcoef(a[l:], b[:-l])[0, 1] for l in lags])
    arr = np.array(curves)
    return lags, np.nanmean(arr, axis=0), np.nanstd(arr, axis=0)


def composite_difference(ds_new, idx_new, ds_ref, idx_ref, var=VAR):
    """Subsurface composite of the top-10% Niño3.4 months, differenced between two runs."""
    out = []
    for ds, idx in [(ds_new, idx_new), (ds_ref, idx_ref)]:
        sub = ds[var].sel(y=slice(-5, 5), x=slice(120, 290), lev=slice(0, 500))
        anomaly = sub.groupby("time.month") - sub.groupby("time.month").mean("time")
        mask = idx.values >= np.percentile(idx.values, 90)
        out.append(anomaly.isel(time=mask).mean(["time", "y"]).compute())
    return out[0] - out[1]


# Each rollout defines its own record; the truth is cut to the same span.
pi_times = xr.open_zarr(ROLLOUT[("pi", "piControl")])["time"].values
ho_times = xr.open_zarr(ROLLOUT[("pi", "Holocene")])["time"].values
spans = {"piControl": slice(pi_times[0], pi_times[-1]),
         "Holocene": slice(ho_times[0], ho_times[-1])}

datasets, indices = {}, {"Nino3.4": {}, "DMI": {}}
for name, path in SERIES.items():
    ds = xr.open_zarr(path, consolidated=True)
    if "year" in ds.dims:
        ds = ds.rename({"year": "time"})
    climate = "Holocene" if name in HO_CONTEXT else "piControl"
    datasets[name] = ds.sel(time=spans[climate])
    sst = datasets[name][VAR].isel(lev=0)
    for index, boxes in INDEX_BOXES.items():
        indices[index][name] = compute_index(sst, boxes)

fig, axd = plt.subplot_mosaic(
    [["nino_pi", "dmi_pi"], ["nino_holo", "dmi_holo"], ["nino_auto", "dmi_auto"],
     ["lag_corr", "legend_panel"], ["depth_emu", "depth_truth"], ["cbar_emu", "cbar_truth"]],
    figsize=(7.5, 9.5), constrained_layout=True,
    gridspec_kw={"height_ratios": [0.7, 0.7, 0.8, 0.8, 1, 0.08]})

for index, (ax_pi, ax_ho, months, ylim, titles, loc) in {
    "Nino3.4": ("nino_pi", "nino_holo", LAST_N_MONTHS, (-4, 5),
                (r"$\bf{A})$ Nino3.4 piControl", r"$\bf{B})$ Nino3.4 midHolocene"), "upper left"),
    "DMI": ("dmi_pi", "dmi_holo", LAST_N_MONTHS // 2, (-2.5, 2.5),
            (r"$\bf{D})$ DMI piControl", r"$\bf{E})$ DMI midHolocene"), "upper right"),
}.items():
    for ax_key, keys, title in [(ax_pi, PI_CONTEXT, titles[0]), (ax_ho, HO_CONTEXT, titles[1])]:
        ax = axd[ax_key]
        for key in keys:
            indices[index][key].isel(time=slice(-months, None)).plot(ax=ax, **STYLES[key])
        ax.set_ylabel("Anomaly (°C)")
        ax.set_xlabel("Year")
        ax.margins(x=0)
        ax.set_ylim(ylim)
        ax.set_title(title)
        ax.legend(loc=loc, ncol=3, fontsize=6)

for index, ax_key, title in [("Nino3.4", "nino_auto", r"$\bf{C})$ Nino3.4 Autocorrelation"),
                             ("DMI", "dmi_auto", r"$\bf{F})$ DMI Autocorrelation")]:
    ax = axd[ax_key]
    for key, style in STYLES.items():
        lags, mean, spread = autocorrelation(indices[index][key])
        ax.plot(lags, mean, **style)
        ax.fill_between(lags, mean - spread, mean + spread, color=style["c"], alpha=0.2)
    ax.set_title(title)
    ax.set_xlabel("Lag (Months)")
    ax.axhline(0, c="k", ls=":", lw=0.8)
    ax.grid(True, alpha=0.3)

ax_lag = axd["lag_corr"]
for key, style in STYLES.items():
    lags, mean, spread = lagged_correlation(indices["Nino3.4"][key], indices["DMI"][key])
    if lags is None:
        continue
    ax_lag.plot(lags, mean, **style)
    ax_lag.fill_between(lags, mean - spread, mean + spread, color=style["c"], alpha=0.2)
ax_lag.set_title(r"$\bf{G})$ Lagged Correlation: Nino3.4 vs DMI")
ax_lag.set_ylabel("Correlation")
ax_lag.set_xlabel("Lag (Months)")
ax_lag.axhline(0, color="k", ls=":", lw=0.8)
ax_lag.axvline(0, color="k", ls=":", lw=0.8)
ax_lag.grid(True, alpha=0.3)

axd["legend_panel"].axis("off")
handles, labels = ax_lag.get_legend_handles_labels()
shading = mpatches.Patch(color="gray", alpha=0.2, label=r"$\pm 1\sigma$")
axd["legend_panel"].legend(handles=handles + [shading], labels=labels + [shading.get_label()],
                           ncol=2, fontsize=10, loc="center", frameon=False)

composites = {
    "emu": composite_difference(datasets["Emu: piC -> Holo"], indices["Nino3.4"]["Emu: piC -> Holo"],
                                datasets["Emu: piC -> piC"], indices["Nino3.4"]["Emu: piC -> piC"]),
    "truth": composite_difference(datasets["Truth Holocene"], indices["Nino3.4"]["Truth Holocene"],
                                  datasets["Truth piControl"], indices["Nino3.4"]["Truth piControl"]),
}
flat = [c.values.flatten() for c in composites.values()]
valid = np.isfinite(flat[0]) & np.isfinite(flat[1])
rho = np.corrcoef(flat[0][valid], flat[1][valid])[0, 1]

for key, ax_key, cax_key, title in [
    ("emu", "depth_emu", "cbar_emu",
     r"$\bf{H})$ Top 10% Nino3.4 Profile" "\n"
     r"$\Omega(\mathcal{F}_{\mathrm{pi}}; \boldsymbol{\tau}_{\mathrm{mH}})$ - "
     r"$\Omega(\mathcal{F}_{\mathrm{pi}}; \boldsymbol{\tau}_{\mathrm{pi}})$" + rf" ($\rho$ = {rho:.2f})"),
    ("truth", "depth_truth", "cbar_truth",
     r"$\bf{I})$ Top 10% Nino3.4 Profile" "\n" r"CESM2 mH - piC"),
]:
    ax, cax, data = axd[ax_key], axd[cax_key], composites[key]
    mesh = ax.pcolormesh(data.x.values, data.lev.values, data.values,
                         cmap="RdBu_r", vmin=-1.0, vmax=1.0, shading="auto")
    ax.set_ylim(500, 0)
    ax.set_ylabel("Depth (m)")
    ax.set_xlabel("Longitude")
    ax.set_title(title, loc="left", fontsize=11)
    plt.colorbar(mesh, cax=cax, orientation="horizontal", label="Temp Diff (°C)")

plt.savefig(f"{FIG_DIR}/Variability_Plot_Depth.png", dpi=300)
plt.show()

## Figure 5 — change in temporal variability

The standard deviation of the state over the last 50 years of each rollout, differenced
between climates, over 0–200 m (Fig. 5) and 200–1000 m (Fig. S27).

The window is defined once per climate by that climate's rollout and then imposed on
every other store, matched on year × month. Taking the last 50 years of each store
independently does not work: the rollouts stop before the truth records do, which would
put the midHolocene truth five years downstream of the prediction it is scored against.

In [ ]:
YEARS_TO_KEEP = 50
VAR_CLIP = 2.25              # a change in sigma, so a much tighter bound than a response
DEPTH_BANDS = [(0, 200, "0-200m"), (200, 1000, "200-1000m")]

CLIMATE_OF = {
    "Truth piControl": "piControl", "Truth Holocene": "midHolocene",
    "Emu: piC -> piC": "piControl", "Emu: piC -> Holo": "midHolocene",
    "Emu: Holo -> Holo": "midHolocene", "Emu: Holo -> piC": "piControl",
}
WINDOW_REFERENCE = {"piControl": "Emu: piC -> piC", "midHolocene": "Emu: piC -> Holo"}
PANELS = [
    ("Truth Diff", r"$\mathbf{A}$) CESM2: midHolo $-$ piC"),
    ("Emu IC Diff", r"$\mathbf{B}$) $\Omega(\mathcal{F}_{\mathrm{mH}};\ \boldsymbol{\tau}_{\mathrm{mH}}) - "
                    r"\Omega(\mathcal{F}_{\mathrm{pi}};\ \boldsymbol{\tau}_{\mathrm{pi}})$"),
    ("Emu piC Resp", r"$\mathbf{C}$) $\Omega(\mathcal{F}_{\mathrm{pi}};\ \boldsymbol{\tau}_{\mathrm{mH}}) - "
                     r"\Omega(\mathcal{F}_{\mathrm{pi}};\ \boldsymbol{\tau}_{\mathrm{pi}})$"),
    ("Emu Holo Resp", r"$\mathbf{D}$) $\Omega(\mathcal{F}_{\mathrm{mH}};\ \boldsymbol{\tau}_{\mathrm{mH}}) - "
                      r"\Omega(\mathcal{F}_{\mathrm{mH}};\ \boldsymbol{\tau}_{\mathrm{pi}})$"),
]
METRIC_PANELS = ["Emu IC Diff", "Emu piC Resp", "Emu Holo Resp"]


def steps_per_year(times):
    """12 for a monthly axis, 1 for an annual one, inferred from the spacing."""
    t = np.asarray(times)
    if t.size < 2:
        return 1
    head = t[:13]
    if np.issubdtype(t.dtype, np.datetime64):
        days = float(np.median(np.diff(head).astype("timedelta64[h]").astype(float))) / 24.0
    elif t.dtype == object and hasattr(head[0], "year"):
        days = float(np.median([(head[i + 1] - head[i]).total_seconds()
                                for i in range(len(head) - 1)])) / 86400.0
    else:
        return 1
    return int(round(365.25 / days)) if days > 0 else 1


def restrict_to_window(ds, m0, m1):
    key = month_key(ds["time"].values)
    idx = np.flatnonzero((key >= m0) & (key <= m1))
    if idx.size == 0:
        raise ValueError("no overlap with the requested window")
    return ds.isel(time=idx)


def band_sigma(ds, band, var=VAR):
    """Temporal standard deviation at each point, dz-averaged over a depth band."""
    field = ds[var].sel(lev=slice(*band))
    dz_band = DZ.sel(lev=field.lev).fillna(0)
    return field.std("time").weighted(dz_band).mean("lev").compute()


def shared_mask(maps, clip=VAR_CLIP):
    """Keep only points finite and within +/- clip in EVERY map, so all are scored alike."""
    aligned = dict(zip(maps, xr.align(*maps.values(), join="inner")))
    keep = None
    for m in aligned.values():
        valid = np.isfinite(m) & (np.abs(m) <= clip)
        keep = valid if keep is None else (keep & valid)
    print(f"    shared mask keeps {int(keep.sum())}/{int(keep.size)} points")
    return {k: v.where(keep) for k, v in aligned.items()}


def masked_score(a, b, weights):
    """Pattern correlation and RMSE with the weights restricted to the scored points.

    `rmse` skips NaN in its numerator but divides by `weights.sum()`, so an unmasked
    areacello against a basin-restricted field deflates every regional RMSE.
    """
    a, b, w = xr.align(a, b, weights, join="inner")
    w = w.where(np.isfinite(a) & np.isfinite(b))
    if float(w.sum()) == 0:
        return np.nan, np.nan
    return corr(a, b, w), rmse(a, b, w)


STORE_OF = {
    "Truth piControl": TRUTH["piControl"],
    "Truth Holocene": TRUTH["Holocene"],
    "Emu: piC -> piC": ROLLOUT[("pi", "piControl")],
    "Emu: piC -> Holo": ROLLOUT[("pi", "Holocene")],
    "Emu: Holo -> Holo": ROLLOUT[("mH", "Holocene")],
    "Emu: Holo -> piC": ROLLOUT[("mH", "piControl")],
}

raw = {}
for name, path in STORE_OF.items():
    ds = xr.open_zarr(path, consolidated=True)
    raw[name] = ds.rename({"year": "time"}) if "year" in ds.dims else ds

windows = {}
for climate, ref_name in WINDOW_REFERENCE.items():
    ref = raw[ref_name]
    n_steps = YEARS_TO_KEEP * steps_per_year(ref["time"].values)
    key = month_key(ref.isel(time=slice(-n_steps, None))["time"].values)
    windows[climate] = (int(key[0]), int(key[-1]))
    print(f"  {climate:<12} window {key[0]} -> {key[-1]} ({n_steps} steps)")

aligned = {name: restrict_to_window(ds, *windows[CLIMATE_OF[name]]) for name, ds in raw.items()}

open_ocean = xr.where(
    (MASKS["Atlantic"].fillna(0) + MASKS["Pacific"].fillna(0) + MASKS["Indian"].fillna(0)) > 0,
    1.0, np.nan)

for z_min, z_max, label in DEPTH_BANDS:
    print(f"\n{label}:")
    sigma = {name: band_sigma(ds, (z_min, z_max)) for name, ds in aligned.items()}
    maps = shared_mask({
        "Truth Diff": sigma["Truth Holocene"] - sigma["Truth piControl"],
        "Emu IC Diff": sigma["Emu: Holo -> Holo"] - sigma["Emu: piC -> piC"],
        "Emu piC Resp": sigma["Emu: piC -> Holo"] - sigma["Emu: piC -> piC"],
        # Defined as Omega(mH;mH) - Omega(mH;pi) so that it, the panel title and the truth
        # all share the midHolocene-minus-piControl orientation.
        "Emu Holo Resp": sigma["Emu: Holo -> Holo"] - sigma["Emu: Holo -> piC"],
    })
    area = AREACELLO.sel(y=maps["Truth Diff"].y, x=maps["Truth Diff"].x)
    metrics = {k: masked_score(maps[k], maps["Truth Diff"], area) for k in METRIC_PANELS}

    for scope, mask in [("global (no Southern)", open_ocean)] + [(b, MASKS[b]) for b in BASINS]:
        line = "  ".join(
            f"{k}: rho={masked_score(maps[k] * mask, maps['Truth Diff'] * mask, area)[0]:.2f}"
            for k in METRIC_PANELS)
        print(f"    {scope:<22} {line}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), subplot_kw={"projection": PROJECTION},
                             constrained_layout=True)
    values = np.concatenate([m.values.flatten() for m in maps.values()])
    vmax = np.percentile(np.abs(values[np.isfinite(values)]), 99)
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    for ax, (key, title) in zip(axes.flat, PANELS):
        data = maps[key]
        mesh = ax.pcolormesh(data["x"], data["y"], data.values, transform=ccrs.PlateCarree(),
                             cmap=cmocean.cm.balance, norm=norm, shading="auto")
        ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=10)
        ax.coastlines(zorder=11)
        ax.set_title(title, loc="left", fontsize=16, pad=8)
        if key in metrics:
            r, rms = metrics[key]
            ax.text(0.03, 0.04, f"vs CESM2\n$\\rho$ = {r:.2f}\nRMSE = {rms:.3f} {UNITS}",
                    transform=ax.transAxes, fontsize=15, va="bottom", ha="left", linespacing=1.35,
                    bbox=dict(facecolor="white", alpha=0.85, edgecolor="lightgray",
                              boxstyle="round,pad=0.4"), zorder=15)
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = gl.ylabel_style = {"size": 11}

    cbar = fig.colorbar(mesh, ax=axes, orientation="horizontal", shrink=0.6, pad=0.03)
    cbar.set_label(f"$\\Delta$ Std. Dev. of potential temperature ({UNITS})", fontsize=17)
    fig.suptitle(f"Change in temporal variability, {label} (last {YEARS_TO_KEEP} yr)", fontsize=22)
    plt.savefig(f"{FIG_DIR}/Variability_Change_{label}.png", dpi=300, bbox_inches="tight")
    plt.show()

## Figure 6 — time-mean response, basin sections

The zonally averaged response in each basin, for both emulators, against the CESM2
difference. Stippling marks where the 95% confidence interval on the mean response
includes zero: five initial conditions for the emulators, ten decadal means for CESM2.

$R(\mathcal{F}_{\mathrm{mH}})$ is scored against the negated truth, per the sign
convention above.

In [ ]:
YEAR_WINDOW = slice(-25, None)
FS = {"title": 14, "label": 14, "tick": 12, "cbar_label": 12, "annotation": 10}

stores = {
    "pi_ctrl": xr.open_zarr(response_path("pi", EPOCH_PI)),
    "pi_full": xr.open_zarr(response_path("pi", EPOCH_PI, FULL_FORCING)),
    "mH_ctrl": xr.open_zarr(response_path("mH", EPOCH_MH)),
    "mH_full": xr.open_zarr(response_path("mH", EPOCH_MH, FULL_FORCING)),
    "truth_pi": xr.open_zarr(TRUTH["piControl"]),
    "truth_ho": xr.open_zarr(TRUTH["Holocene"]),
}

sections, spreads, counts, metrics = {}, {}, {}, {}
for basin in BASINS:
    mask = basin_mask(basin)
    lo, hi = basin_bounds(mask)
    truth = zonal_mean_response(stores["truth_pi"], stores["truth_ho"], mask,
                                window=YEAR_WINDOW, is_truth=True)
    chunks, n_truth = truth_chunk_means(stores["truth_pi"], stores["truth_ho"],
                                        max_depth=MAX_DEPTH)
    truth_spread = zonal_profile(chunks, mask, wet_mask(stores["truth_pi"])).std(
        "chunk").sel(y=slice(lo, hi)).compute()

    members = {
        "pi_to_holo": zonal_mean_response(stores["pi_ctrl"], stores["pi_full"], mask,
                                          window=YEAR_WINDOW, collapse=False),
        "holo_to_pi": zonal_mean_response(stores["mH_ctrl"], stores["mH_full"], mask,
                                          window=YEAR_WINDOW, collapse=False),
    }
    entry = {"truth": truth, "truth_spread": truth_spread, "n_truth": n_truth}
    for key, field in members.items():
        entry[key], entry[f"{key}_spread"], entry["n_emu"] = ensemble_stats(field)
    sections[basin] = entry

    area = section_area()
    metrics[basin] = {
        "pi_to_holo": score(entry["pi_to_holo"], truth, area),
        "holo_to_pi": score(entry["holo_to_pi"], -truth, area),
        "symmetry": corr(entry["pi_to_holo"], -entry["holo_to_pi"], area),
    }
    m = metrics[basin]
    print(f"  {basin:<9} pi->mH rho={m['pi_to_holo']['rho']:.2f} amp={m['pi_to_holo']['amp']:.2f} | "
          f"mH->pi rho={m['holo_to_pi']['rho']:.2f} amp={m['holo_to_pi']['amp']:.2f} | "
          f"symmetry rho={m['symmetry']:.2f}")

fig, axes = plt.subplots(4, 3, figsize=(16, 10), constrained_layout=True)
keys = ["truth", "pi_to_holo", "holo_to_pi"]

for i, basin in enumerate(BASINS):
    entry, m = sections[basin], metrics[basin]
    vmax = max(np.abs(entry[k]).max().item() for k in keys)
    norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)
    area = section_area()

    for j, key in enumerate(keys):
        ax = axes[i, j]
        mesh = entry[key].plot.contourf(ax=ax, cmap=cmocean.cm.balance, norm=norm,
                                        add_colorbar=False, levels=25, yincrease=False,
                                        extend="both")
        n = entry["n_truth"] if key == "truth" else entry["n_emu"]
        stipple_section(ax, insignificance_mask(entry[key], entry[f"{key}_spread"], n))
        if j == 0:
            first = mesh
    cbar = fig.colorbar(first, ax=axes[i, :], orientation="vertical", shrink=0.8, pad=0.02)
    cbar.set_label(f"Response ({UNITS})", fontsize=FS["cbar_label"])
    axes[i, 0].set_ylabel(f"{basin}\n\nDepth (m)", fontsize=FS["label"])
    for ax in axes[i, 1:]:
        ax.set_ylabel("")

    xlim, ylim = axes[i, 0].get_xlim(), axes[i, 0].get_ylim()
    for region in [r for r in METRIC_REGIONS if r["basin"] == basin]:
        sub = dict(y=slice(*region["lat"]), lev=slice(*region["depth"]))
        truth_sub, area_sub = entry["truth"].sel(**sub), area.sel(**sub)
        if truth_sub.size == 0 or float(area_sub.sum()) == 0:
            continue
        region_scores = {
            "pi_to_holo": score(entry["pi_to_holo"].sel(**sub), truth_sub, area_sub),
            "holo_to_pi": score(entry["holo_to_pi"].sel(**sub), -truth_sub, area_sub),
        }
        x0 = max(region["lat"][0], xlim[0])
        y0 = max(region["depth"][0], ylim[1])
        width = min(region["lat"][1], xlim[1]) - x0
        height = min(region["depth"][1], ylim[0]) - y0
        if width <= 0 or height <= 0:
            continue
        for j, ax in enumerate(axes[i, :]):
            ax.add_patch(plt.Rectangle((x0, y0), width, height, linewidth=2.0,
                                       edgecolor=region["color"], linestyle="--",
                                       facecolor="none", zorder=10, clip_on=False))
            if j == 0:
                text = r"$\mathbf{" + region["name"].replace(" ", r"\ ") + r"}$"
            else:
                s = region_scores[keys[j]]
                text = r"$\rho=$" + f"{s['rho']:.2f}\nRMSE={s['rmse']:.2f}"
            ax.text(x0 + 0.02 * width, y0 + height - 0.02 * height, " " + text, color="black",
                    ha="left", va="bottom", fontsize=FS["annotation"], zorder=12,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.2,
                              ec=region["color"], lw=1.5))

for k, ax in enumerate(axes.flat):
    i, j = divmod(k, 3)
    basin, m = BASINS[i], metrics[BASINS[i]]
    ax.set_xlabel("")
    ax.grid(True, linestyle=":", alpha=0.6)
    linear_piecewise_scale(500, 2, ax=ax)
    ax.tick_params(axis="both", labelsize=FS["tick"])
    title = panel_letter(k) + " "
    if i == 0:
        title += RESPONSE_COLUMN_TITLES[j]
    if j > 0:
        s = m[keys[j]]
        sep = "\n" if i == 0 else " "
        title += f"{sep}(vs {string.ascii_uppercase[i * 3]}: $\\rho$={s['rho']:.2f}, RMSE={s['rmse']:.2f})"
    ax.set_title(title, loc="left", fontsize=FS["title"])
for ax in axes[-1, :]:
    ax.set_xlabel("Latitude", fontsize=FS["label"])

plt.savefig(f"{FIG_DIR}/Responses_Full_Comparisons_Metrics_{VAR}.png", dpi=300, bbox_inches="tight")
plt.show()

## Figure 7 — time-mean response, depth-averaged maps

The same responses as maps, averaged over 0–200 m and 200–1000 m, with the same
significance stippling.

In [ ]:
DEPTH_SLICES = [(0, 200), (200, 1000)]
YEAR_WINDOW = slice(-20, None)
FS = {"title": 11, "label": 10}

stores = {
    "pi_ctrl": xr.open_zarr(response_path("pi", EPOCH_PI)),
    "pi_full": xr.open_zarr(response_path("pi", EPOCH_PI, FULL_FORCING)),
    "mH_ctrl": xr.open_zarr(response_path("mH", EPOCH_MH)),
    "mH_full": xr.open_zarr(response_path("mH", EPOCH_MH, FULL_FORCING)),
    "truth_pi": xr.open_zarr(TRUTH["piControl"]),
    "truth_ho": xr.open_zarr(TRUTH["Holocene"]),
}


def response_maps(ctrl, pert, is_truth=False):
    """Depth-averaged response per band, with the ensemble mean and spread."""
    time_dim = "time" if is_truth else "year"
    window = slice(-100 * 12, None) if is_truth else YEAR_WINDOW
    field = (pert[VAR] - ctrl[VAR]).isel({time_dim: window}).mean(time_dim).compute()
    out = {}
    for band in DEPTH_SLICES:
        mean, spread, n = ensemble_stats(depth_mean(field, band))
        out[band] = {"mean": mean.compute(), "spread": spread.compute(), "n": n}
    return out


truth_maps = response_maps(stores["truth_pi"], stores["truth_ho"], is_truth=True)
pi_maps = response_maps(stores["pi_ctrl"], stores["pi_full"])
mH_maps = response_maps(stores["mH_ctrl"], stores["mH_full"])

truth_chunks, n_truth = truth_chunk_means(stores["truth_pi"], stores["truth_ho"])
truth_chunks = truth_chunks.compute()
truth_spread = {band: depth_mean(truth_chunks, band).std("chunk").compute()
                for band in DEPTH_SLICES}

responses, metrics = {}, {}
for band in DEPTH_SLICES:
    truth = truth_maps[band]["mean"]
    responses[band] = {
        "truth": truth, "truth_spread": truth_spread[band], "truth_n": n_truth,
        "pi_to_holo": pi_maps[band]["mean"], "pi_to_holo_spread": pi_maps[band]["spread"],
        "pi_to_holo_n": pi_maps[band]["n"],
        "holo_to_pi": mH_maps[band]["mean"], "holo_to_pi_spread": mH_maps[band]["spread"],
        "holo_to_pi_n": mH_maps[band]["n"],
    }
    metrics[band] = {"pi_to_holo": score(pi_maps[band]["mean"], truth, AREACELLO),
                     "holo_to_pi": score(mH_maps[band]["mean"], -truth, AREACELLO)}
    m = metrics[band]
    print(f"  {band[0]}-{band[1]} m  pi->mH rho={m['pi_to_holo']['rho']:.2f} "
          f"RMSE={m['pi_to_holo']['rmse']:.3f} | mH->pi rho={m['holo_to_pi']['rho']:.2f} "
          f"RMSE={m['holo_to_pi']['rmse']:.3f}")

layout = [[f"ax_{i}{j}" for j in range(3)] + [f"cbar_{i}"] for i in range(len(DEPTH_SLICES))]
fig, axes = plt.subplot_mosaic(
    layout, figsize=(12, 2.5 * len(DEPTH_SLICES)), layout="constrained",
    per_subplot_kw={key: {"projection": PROJECTION}
                    for row in layout for key in row[:-1]},
    gridspec_kw={"width_ratios": [1, 1, 1, 0.05]})

keys = ["truth", "pi_to_holo", "holo_to_pi"]
for i, band in enumerate(DEPTH_SLICES):
    entry, m = responses[band], metrics[band]
    vmax = max(np.abs(entry[k]).quantile(0.99).item() for k in keys)
    norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)

    for j, key in enumerate(keys):
        ax, data = axes[f"ax_{i}{j}"], entry[key]
        kwargs = dict(ax=ax, transform=ccrs.PlateCarree(), cmap=cmocean.cm.balance,
                      norm=norm, x="lon", y="lat")
        if j == 0:
            data.plot.pcolormesh(add_colorbar=True, cbar_ax=axes[f"cbar_{i}"],
                                 cbar_kwargs={"label": f"Response ({UNITS})"}, **kwargs)
        else:
            data.plot.pcolormesh(add_colorbar=False, **kwargs)
        stipple_map(ax, data["lon"], data["lat"],
                    insignificance_mask(data, entry[f"{key}_spread"], entry[f"{key}_n"]))
        ax.add_feature(cfeature.LAND, zorder=2, edgecolor="black", facecolor="lightgray")
        ax.coastlines(zorder=2)
        gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                          alpha=0.5, linestyle=":")
        gl.top_labels = gl.right_labels = False
        gl.bottom_labels = i == len(DEPTH_SLICES) - 1
        gl.left_labels = j == 0

        title = RESPONSE_COLUMN_TITLES[j]
        if key != "truth":
            reference = "A" if i == 0 else "D"
            title += f"\nvs {reference}: ($\\rho$={m[key]['rho']:.2f}, RMSE={m[key]['rmse']:.3f})"
        ax.set_title(title, fontsize=FS["title"])
        ax.set_title(panel_letter(i * 3 + j), loc="left", fontsize=FS["title"])

    axes[f"ax_{i}0"].text(-0.25, 0.5, f"{VAR_SYMBOL}: {band[0]}-{band[1]} m",
                          transform=axes[f"ax_{i}0"].transAxes, fontsize=FS["label"],
                          rotation="vertical", va="center", ha="left")

plt.savefig(f"{FIG_DIR}/Responses_Full_Maps_Multi_Depth_Comparisons_{VAR}.png",
            dpi=300, bbox_inches="tight")
plt.show()

## Figure 8 — responses to individual forcing components

The full emulator perturbed one boundary component at a time (top row of each pair)
against an emulator trained on that component alone (bottom row). "Symmetry ρ" is the
correlation between the two, i.e. how far the full emulator's component response agrees
with a model that only ever saw that forcing.

Set `BASIN` to `Pacific`, `Atlantic` or `Indian` for the supplementary versions.

In [ ]:
BASIN = "Southern"
YEAR_WINDOW = slice(-20, None)
COLORBAR_RATIO = 0.8
FS = {"panel_title": 14, "axis_label": 16, "tick": 14, "cbar_label": 16, "cbar_tick": 14,
      "text_box": 10}

mask = basin_mask(BASIN)
control = xr.open_zarr(response_path("pi", EPOCH_PI))

responses = {"truth": zonal_mean_response(xr.open_zarr(TRUTH["piControl"]),
                                          xr.open_zarr(TRUTH["Holocene"]), mask,
                                          window=YEAR_WINDOW, is_truth=True)}
for forcings, key in [(FULL_FORCING, "generalist_full"), (["hfds"], "generalist_hfds"),
                      (["tauuo"], "generalist_tauuo"), (["tauvo"], "generalist_tauvo")]:
    responses[key] = zonal_mean_response(
        control, xr.open_zarr(response_path("pi", EPOCH_PI, forcings)), mask, window=YEAR_WINDOW)

for forcing, base in SPECIALIST.items():
    responses[f"specialist_{forcing}"] = zonal_mean_response(
        xr.open_zarr(f"{base}/yearly_means.zarr"),
        xr.open_zarr(f"{base}_{forcing}/yearly_means.zarr"), mask, window=YEAR_WINDOW)

area = section_area().mean("x")
metrics = {k: score(v, responses["truth"], area) for k, v in responses.items() if k != "truth"}
symmetry = {f: corr(responses[f"specialist_{f}"], responses[f"generalist_{f}"], area)
            for f in SPECIALIST}

mosaic = [["A", "C", "E", "G", "cbar"], ["B", "D", "F", "H", "cbar"]]
fig, axes = plt.subplot_mosaic(mosaic, figsize=(18, 8), constrained_layout=True,
                               gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.05],
                                            "height_ratios": [1, 1]})
cmap = cmocean.cm.balance.copy()
cmap.set_bad(color="grey")
vmax = np.abs(responses["truth"]).max().item() * COLORBAR_RATIO
norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)

panel_of = {"generalist_full": "A", "truth": "B", "generalist_hfds": "C", "specialist_hfds": "D",
            "generalist_tauuo": "E", "specialist_tauuo": "F", "generalist_tauvo": "G",
            "specialist_tauvo": "H"}
titles = {
    "generalist_full": r"$\mathcal{E}(\mathcal{F}_{pi}^{\mathrm{full}},\boldsymbol{\tau}_{\mathrm{mH}},I_\mathrm{mH}) - "
                       r"\mathcal{E}(\mathcal{F}_{pi}^{\mathrm{full}},\boldsymbol{\tau}_{\mathrm{pi}})$",
    "truth": "CESM2: midHolo - piC",
    "generalist_hfds": r"$\mathcal{E}(\mathcal{F}_{pi}^{\mathrm{full}};\overline{\mathrm{hfds}}_{\mathrm{mH}}) - "
                       r"\mathcal{E}(\mathcal{F}_{pi}^{\mathrm{full}};\overline{\boldsymbol{\tau}}_{\mathrm{pi}})$",
    "specialist_hfds": r"$\mathcal{E}(\mathcal{F}_{\mathrm{hfds}};\overline{\mathrm{hfds}}_{\mathrm{mH}}) - "
                       r"\mathcal{E}(\mathcal{F}_{\mathrm{hfds}};\overline{\mathrm{hfds}}_{\mathrm{pi}})$",
    "generalist_tauuo": r"$\mathcal{E}(\mathcal{F}_{pi}^{\mathrm{full}};\overline{\tau_u}_{\mathrm{mH}}) - "
                        r"\mathcal{E}(\mathcal{F}_{pi}^{\mathrm{full}};\overline{\boldsymbol{\tau}}_{\mathrm{pi}})$",
    "specialist_tauuo": r"$\mathcal{E}(\mathcal{F}_{\tau_{u}};[\overline{\tau_{u}}]_{\mathrm{mH}}) - "
                        r"\mathcal{E}(\mathcal{F}_{\tau_{u}};[\overline{\tau_{u}}]_{\mathrm{pi}})$",
    "generalist_tauvo": r"$\mathcal{E}(\mathcal{F}_{pi}^{\mathrm{full}};\overline{\tau_v}_{\mathrm{mH}}) - "
                        r"\mathcal{E}(\mathcal{F}_{pi}^{\mathrm{full}};\overline{\boldsymbol{\tau}}_{\mathrm{pi}})$",
    "specialist_tauvo": r"$\mathcal{E}(\mathcal{F}_{pi}^{\tau_{v}};[\overline{\tau_{v}}]_{\mathrm{mH}}) - "
                        r"\mathcal{E}(\mathcal{F}_{pi}^{\tau_{v}};[\overline{\tau_{v}}]_{\mathrm{pi}})$",
}

for key, panel in panel_of.items():
    ax = axes[panel]
    mesh = responses[key].plot.contourf(ax=ax, cmap=cmap, norm=norm, add_colorbar=False,
                                        levels=25, yincrease=False, extend="both")
    title = r"$\mathbf{" + panel + r"}$) " + titles[key]
    if key in metrics:
        title += f"\n(vs B): $\\rho$={metrics[key]['rho']:.2f}, RMSE={metrics[key]['rmse']:.3f})"
    ax.set_title(title, loc="left", fontsize=FS["panel_title"])

for forcing, panel in [("hfds", "C"), ("tauuo", "E"), ("tauvo", "G")]:
    axes[panel].text(0.5, -0.25, rf"Symmetry $\rho$ = {symmetry[forcing]:.2f}",
                     ha="center", va="center", transform=axes[panel].transAxes,
                     fontsize=FS["text_box"],
                     bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=1))
axes["A"].text(0.5, -0.25,
               r"$\mathrm{Unless~ Specified:}$" "\n"
               r"$\boldsymbol{\Phi} = \boldsymbol{\Phi}_{\mathcal{F}_{pi}}^{[0]}~,"
               r"\overline{\boldsymbol{\tau}}=[\overline{\operatorname{hfds}}_{pi},"
               r"[\overline{\tau_u}]_{pi},[\overline{\tau_v}]_{pi}],~I=I_{pi}$",
               ha="center", va="center", transform=axes["A"].transAxes, fontsize=12,
               bbox=dict(boxstyle="round,pad=0.75", fc="white", ec="black", lw=1))

cbar = fig.colorbar(mesh, cax=axes["cbar"], orientation="vertical", extend="both")
cbar.set_label(f"Response ({UNITS})", fontsize=FS["cbar_label"])
cbar.ax.tick_params(labelsize=FS["cbar_tick"])

xlim, ylim = axes["A"].get_xlim(), axes["A"].get_ylim()
for panel, ax in axes.items():
    if panel == "cbar":
        continue
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_facecolor("grey")
    ax.grid(True, linestyle=":", alpha=0.6)
    linear_piecewise_scale(500, 2, ax=ax)
    ax.tick_params(axis="both", labelsize=FS["tick"])
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    if panel in ["A", "B"]:
        ax.set_ylabel("Depth (m)", fontsize=FS["axis_label"])
    else:
        ax.set_yticklabels([])
    if panel in ["B", "D", "F", "H"]:
        ax.set_xlabel("Latitude", fontsize=FS["axis_label"])
    else:
        ax.set_xticklabels([])

region_text = {p: [] for p in "ACEGDFH"}
for region in [r for r in METRIC_REGIONS if r["basin"] == BASIN]:
    sub = dict(y=slice(*region["lat"]), lev=slice(*region["depth"]))
    truth_sub, area_sub = responses["truth"].sel(**sub), area.sel(**sub)
    if truth_sub.size == 0 or float(area_sub.sum()) == 0:
        continue
    for key, panel in panel_of.items():
        if key == "truth":
            continue
        region_text[panel].append(
            r"$\mathbf{" + region["name"].replace(" ", r"\ ") + r":}\ \rho = "
            + f"{corr(responses[key].sel(**sub), truth_sub, area_sub):.2f}$")

    x0 = max(region["lat"][0], xlim[0])
    y0 = max(region["depth"][0], ylim[1])
    width = min(region["lat"][1], xlim[1]) - x0
    height = min(region["depth"][1], ylim[0]) - y0
    if width <= 0 or height <= 0:
        continue
    for i, panel in enumerate("ABCDEFGH"):
        axes[panel].add_patch(plt.Rectangle((x0, y0), width, height, linewidth=2.5,
                                            edgecolor=region["color"], linestyle="--",
                                            facecolor="none", zorder=10, clip_on=False))
        if i == 0:
            axes[panel].text(x0, y0 + height * 0.95, " " + region["name"], color=region["color"],
                             ha="left", va="bottom", fontsize=FS["text_box"], fontweight="bold",
                             zorder=11, clip_on=False)

for panel, lines in region_text.items():
    if lines:
        axes[panel].text(0.02, 0.05, "\n".join(lines), transform=axes[panel].transAxes,
                         fontsize=FS["text_box"], va="bottom", ha="left",
                         bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.2,
                                   ec="black", lw=1.5))

plt.savefig(f"{FIG_DIR}/Responses_Specialist_Comparisons_Updated_{VAR}_{BASIN}.png",
            dpi=300, bbox_inches="tight")
plt.show()

## Figure 9 — additivity of the component responses

Each panel is a residual: a component response, or a sum of them, minus the full
response. A residual small against the total response means the emulator combines
forcings close to linearly. This is a statement about the emulator, not about CESM2 —
single-forcing numerical runs are not part of the PMIP4 protocol.

In [ ]:
RESIDUAL_BASINS = ["Southern", "Pacific", "Atlantic"]
YEAR_WINDOW = slice(-25, None)
CBAR_RATIO = 0.9
FS = {"suptitle": 22, "title": 14, "tick": 12, "cbar_label": 12}

COMPONENTS = ["hfds", "tauuo", "tauvo", "orb"]
RESIDUAL_TITLES = {
    "truth": "CESM2 Truth (midHolocene - piControl)",
    "full": r"Total Emulator Response ($\mathcal{E}_{\mathrm{full}}$)",
    "res_hfds": "Residual: HFDS Component",
    "res_tauuo": r"Residual: $\tau_u$ Component",
    "res_tauvo": r"Residual: $\tau_v$ Component",
    "res_sum_wind": r"Residual: Wind Sum ($\tau_u$ + $\tau_v$)",
    "res_sum_3": r"Residual: Total Sum (HFDS + $\tau_u$ + $\tau_v$)",
    "res_sum_4": r"Residual: Total Sum (HFDS + $\tau_u$ + $\tau_v$ + Orb)",
}

control = xr.open_zarr(response_path("pi", EPOCH_PI))
perturbed = {"full": xr.open_zarr(response_path("pi", EPOCH_PI, FULL_FORCING))}
perturbed.update({f: xr.open_zarr(response_path("pi", EPOCH_PI, [f])) for f in COMPONENTS})
truth_stores = (xr.open_zarr(TRUTH["piControl"]), xr.open_zarr(TRUTH["Holocene"]))

for basin in RESIDUAL_BASINS:
    mask = basin_mask(basin)
    area = section_area()

    truth = zonal_mean_response(*truth_stores, mask, window=YEAR_WINDOW, is_truth=True)
    full = zonal_mean_response(control, perturbed["full"], mask, window=YEAR_WINDOW)
    parts = {f: zonal_mean_response(control, perturbed[f], mask, window=YEAR_WINDOW)
             for f in COMPONENTS}
    parts["sum_wind"] = parts["tauuo"] + parts["tauvo"]
    parts["sum_3"] = parts["hfds"] + parts["tauuo"] + parts["tauvo"]
    parts["sum_4"] = parts["sum_3"] + parts["orb"]

    keys = ["hfds", "tauuo", "tauvo", "sum_wind", "sum_3", "sum_4"]
    panels = {"truth": truth, "full": full}
    panels.update({f"res_{k}": parts[k] - full for k in keys})

    print(f"\n{basin}:")
    for k in keys:
        print(f"    {k:<9} vs full: rho={corr(parts[k], full, area):.2f}  "
              f"RMSE={rmse(parts[k], full, area):.3f} | vs truth: "
              f"rho={corr(parts[k], truth, area):.2f}  RMSE={rmse(parts[k], truth, area):.3f}")

    mosaic = [["truth", "full", "cbar_abs"],
              ["res_hfds", "res_tauuo", "cbar_res"],
              ["res_tauvo", "res_sum_wind", "cbar_res"],
              ["res_sum_3", "res_sum_4", "cbar_res"]]
    fig, axes = plt.subplot_mosaic(mosaic, figsize=(14, 16), constrained_layout=True,
                                   gridspec_kw={"width_ratios": [1, 1, 0.05]})
    cmap = cmocean.cm.balance.copy()
    cmap.set_bad(color="grey")

    absolute = ["truth", "full"]
    residual = [k for k in panels if k not in absolute]
    vmax_abs = max(np.abs(panels[k]).max().item() for k in absolute) * CBAR_RATIO
    vmax_res = max(np.abs(panels[k]).max().item() for k in residual) * CBAR_RATIO
    norm_abs = mcolors.Normalize(vmin=-vmax_abs, vmax=vmax_abs)
    norm_res = mcolors.Normalize(vmin=-vmax_res, vmax=vmax_res)

    for i, key in enumerate([k for row in mosaic for k in row if not k.startswith("cbar")]):
        ax = axes[key]
        mesh = panels[key].plot.contourf(
            ax=ax, cmap=cmap, norm=norm_abs if key in absolute else norm_res,
            add_colorbar=False, levels=25, yincrease=False, extend="both")
        if key == "truth":
            mesh_abs = mesh
        if key == "res_hfds":
            mesh_res = mesh
        title = rf"$\mathbf{{{string.ascii_uppercase[i]}}}$) {RESIDUAL_TITLES[key]}"
        if key not in absolute:
            component = key[4:]
            title += (f"\n(Component RMSE vs Truth: {rmse(parts[component], truth, area):.3f}"
                      f" | Residual RMS: {rmse(panels[key], 0, area):.3f})")
        ax.set_title(title, loc="left", fontsize=FS["title"])

    ylim, xlim = axes["truth"].get_ylim(), axes["truth"].get_xlim()
    for key, ax in axes.items():
        if key.startswith("cbar"):
            continue
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_facecolor("grey")
        ax.grid(True, linestyle=":", alpha=0.6)
        linear_piecewise_scale(500, 2, ax=ax)
        ax.tick_params(axis="both", labelsize=FS["tick"])
        ax.set_ylim(ylim)
        ax.set_xlim(xlim)

    for cax, mesh, label in [(axes["cbar_abs"], mesh_abs, f"Absolute Response ({UNITS})"),
                             (axes["cbar_res"], mesh_res, f"Residual from Total ({UNITS})")]:
        cbar = fig.colorbar(mesh, cax=cax, orientation="vertical", extend="both")
        cbar.set_label(label, fontsize=FS["cbar_label"])

    fig.suptitle(f"{basin} Ocean: Emulator Component Residuals vs. Total Response",
                 fontsize=FS["suptitle"])
    plt.savefig(f"{FIG_DIR}/Residual_Analysis_{VAR}_{basin}.png", dpi=300, bbox_inches="tight")
    plt.show()

## Figure 10 — response skill across epochs and seeds

Ensemble spread and response skill against the emulator's own test RMSE, for every
checkpoint of four training runs — two seeds per emulator. Skill is the area-weighted
correlation between the zonally averaged emulator and CESM2 response profiles over the
upper 1000 m, the same quantity as Figure 6.

Epochs 10, 15 and 65 are dropped: the first two are far from converged, and 65 is the
unstable midHolocene checkpoint the paper excludes.

Checkpoints from one run are not independent samples, so the Spearman coefficients and
their p-values are descriptive of the checkpoints rather than a significance test. This
is the expensive cell: it opens one rollout and two response ensembles per checkpoint.

In [ ]:
EPOCHS = [20, 25, 30, 35, 40, 45, 47, 50, 55, 57, 60, 62, 67, 70, 72, 75]
UPPER_OCEAN_DEPTH = 1000
YEAR_WINDOW = slice(-25, None)
SCATTER_BASINS = ["Atlantic", "Pacific"]
CORR_METHOD = "spearman"

# Order matters: markers, trend colours and line styles are assigned from the pools below
# in this order.
EXPERIMENTS = [
    {"name": "Reconstructed_Baseline", "base_climate": "piControl", "target_climate": "Holocene",
     "label": r"$\mathcal{F}_{\mathrm{pi}}$ (seed 1)"},
    {"name": "Reconstructed_Baseline_Second_Seed", "base_climate": "piControl",
     "target_climate": "Holocene", "label": r"$\mathcal{F}_{\mathrm{pi}}$ (seed 2)"},
    {"name": "Reconstructed_Holocene", "base_climate": "Holocene", "target_climate": "piControl",
     "label": r"$\mathcal{F}_{\mathrm{mH}}$ (seed 1)"},
    {"name": "Reconstructed_Holocene_Seed_2", "base_climate": "Holocene",
     "target_climate": "piControl", "label": r"$\mathcal{F}_{\mathrm{mH}}$ (seed 2)"},
]
MARKER_POOL = ["o", "s", "^", "D", "v", "p", "*", "h", "X", "<", ">"]
COLOR_POOL = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]
LINESTYLE_POOL = ["-", "--", "-.", ":"]

# 1.0/0.0 masks here, not 1.0/NaN: these are used with `.where()` and in sums.
SCATTER_MASKS = {name: xr.where(xr.open_dataset(PATH_MASKS).basin == code, 1.0, 0.0)
                 for name, code in [("Atlantic", 2), ("Pacific", 3)]}


def rollout_rmse(name, epoch, base_climate):
    """Area- and depth-weighted RMSE of the 100-yr rollout against CESM2 over the same months."""
    path = os.path.join(BASE_PATH_EMU, f"BVP_{name}_{base_climate}_IC_Rollout_Epoch_{epoch}.zarr")
    emu = xr.open_zarr(path, consolidated=True).rename({"time": "month"})
    truth = xr.open_zarr(TRUTH[base_climate], consolidated=True).rename({"time": "month"})
    truth = truth.sel(month=slice(emu["month"][0].values, emu["month"][-1].values))
    bias = (emu[VAR].sel(lev=slice(0, UPPER_OCEAN_DEPTH))
            - truth[VAR].sel(lev=slice(0, UPPER_OCEAN_DEPTH)))
    weights = AREACELLO * DZ.sel(lev=slice(0, UPPER_OCEAN_DEPTH))
    return float(np.sqrt((bias ** 2 * weights).sum(("x", "y", "lev")).mean("month")
                         / weights.sum(("x", "y", "lev"))).values)


def response_spread_and_skill(name, epoch, base_climate, target_climate, mask):
    """Ensemble spread of the zonal-mean response, and its skill against CESM2.

    Skill is the correlation of the zonally averaged profiles over the upper
    `UPPER_OCEAN_DEPTH` metres, matching Figure 6 rather than a 3-D correlation.
    """
    folder = f"BVP_CESM2_{name}_{base_climate}_Epoch_{epoch}_Response_{target_climate}_BC"
    ctrl = xr.open_zarr(os.path.join(BASE_PATH_EMU, folder, "yearly_means.zarr"))
    pert = xr.open_zarr(os.path.join(BASE_PATH_EMU, f"{folder}_hfds_tauuo_tauvo_orb",
                                     "yearly_means.zarr"))
    pert = pert.assign_coords(init_time=ctrl.init_time)
    members = (pert[VAR] - ctrl[VAR]).isel(year=YEAR_WINDOW)

    upper = members.mean("year").sel(lev=slice(0, UPPER_OCEAN_DEPTH)).where(mask)
    dx_masked = DX.where(mask)
    spread_profile = ((upper * dx_masked).sum("x", skipna=True)
                      / dx_masked.sum("x", skipna=True)).std("init_time")
    dz_upper = DZ.sel(lev=slice(0, UPPER_OCEAN_DEPTH))
    weights_2d = dz_upper * (AREACELLO * mask).sum("x", skipna=True)
    spread = float(((spread_profile * weights_2d).sum()
                    / weights_2d.where(spread_profile.notnull()).sum()).values)

    truth_ctrl = xr.open_zarr(TRUTH[base_climate])
    truth_pert = xr.open_zarr(TRUTH[target_climate])
    truth = ((truth_pert[VAR] - truth_ctrl[VAR]).isel(time=slice(-1200, None)).mean("time")
             .sel(lev=slice(0, UPPER_OCEAN_DEPTH)))
    emu = members.mean(("year", "init_time")).sel(lev=slice(0, UPPER_OCEAN_DEPTH))

    # Wet masks so each zonal mean is normalised by wet width only, as in Figure 6.
    wet_emu = wet_mask(ctrl, time_dim="year").sel(lev=slice(0, UPPER_OCEAN_DEPTH))
    wet_truth = wet_mask(truth_ctrl).sel(lev=slice(0, UPPER_OCEAN_DEPTH))
    den_emu = (DX * mask * wet_emu).sum("x", skipna=True)
    den_truth = (DX * mask * wet_truth).sum("x", skipna=True)
    profile_emu = ((emu * DX * mask).sum("x", skipna=True) / den_emu).where(den_emu > 0)
    profile_truth = ((truth * DX * mask).sum("x", skipna=True) / den_truth).where(den_truth > 0)

    valid = profile_emu.notnull() & profile_truth.notnull()
    area = (dz_upper * DX.mean("x")).where(valid)
    skill = corr(profile_emu.where(valid), profile_truth.where(valid), area)
    return spread, skill


def truth_spread(base_climate, target_climate, mask, chunk_years=25, n_chunks=10):
    """Internal-variability reference: the spread across `n_chunks` chunks of the CESM2 pair."""
    resp = (xr.open_zarr(TRUTH[target_climate])[VAR] - xr.open_zarr(TRUTH[base_climate])[VAR])
    chunk_size = chunk_years * 12
    chunks = (resp.isel(time=slice(-n_chunks * chunk_size, None))
              .coarsen(time=chunk_size, boundary="trim")
              .construct({"time": ("chunk", "month_in_chunk")}).mean("month_in_chunk"))
    upper = chunks.sel(lev=slice(0, UPPER_OCEAN_DEPTH)).where(mask)
    dx_masked = DX.where(mask)
    profile = ((upper * dx_masked).sum("x", skipna=True)
               / dx_masked.sum("x", skipna=True)).std("chunk")
    weights_2d = DZ.sel(lev=slice(0, UPPER_OCEAN_DEPTH)) * (AREACELLO * mask).sum("x", skipna=True)
    return float(((profile * weights_2d).sum() / weights_2d.where(profile.notnull()).sum()).values)


def format_pvalue(p):
    if p < 0.01:
        return "p < 0.01"
    if p < 0.05:
        return "p < 0.05"
    return f"p = {p:.2f}"


def plot_trend(ax, x, y, method=CORR_METHOD, color="gray", linestyle="-"):
    """Draw a least-squares trendline and return its rank-correlation annotation."""
    if len(x) < 3:
        return None
    if method == "pearson":
        coefficient, p_value = stats.pearsonr(x, y)
        symbol = "r"
    else:
        coefficient, p_value = stats.spearmanr(x, y)
        symbol = r"\rho"
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(min(x), max(x), 100)
    ax.plot(x_line, slope * x_line + intercept, color=color, linestyle=linestyle,
            linewidth=2.5, zorder=4, alpha=0.8)
    return f"${symbol}$ = {coefficient:.2f} ({format_pvalue(p_value)})"


results, truth_spreads = {}, {}
for experiment in EXPERIMENTS:
    name = experiment["name"]
    base, target = experiment["base_climate"], experiment["target_climate"]
    transition = f"{base}_to_{target}"
    if transition not in truth_spreads:
        truth_spreads[transition] = {b: truth_spread(base, target, SCATTER_MASKS[b])
                                     for b in SCATTER_BASINS}

    entry = {"rmse": [], "valid_epochs": [], "base_climate": base, "target_climate": target,
             "metrics": {b: {"spread": [], "correlation": []} for b in SCATTER_BASINS}}
    for epoch in EPOCHS:
        try:
            rmse_val = rollout_rmse(name, epoch, base)
            per_basin = {b: response_spread_and_skill(name, epoch, base, target, SCATTER_MASKS[b])
                         for b in SCATTER_BASINS}
        except Exception as exc:                       # checkpoints that were never rolled out
            print(f"  skipping {name} epoch {epoch}: {type(exc).__name__}")
            continue
        entry["valid_epochs"].append(epoch)
        entry["rmse"].append(rmse_val)
        for b, (spread, skill) in per_basin.items():
            entry["metrics"][b]["spread"].append(spread)
            entry["metrics"][b]["correlation"].append(skill)
    if entry["valid_epochs"]:
        results[name] = entry
        print(f"  {name}: {len(entry['valid_epochs'])} checkpoints, "
              f"RMSE {min(entry['rmse']):.3f}-{max(entry['rmse']):.3f} {UNITS}")

styled = [dict(experiment, marker=MARKER_POOL[i % len(MARKER_POOL)],
               color=COLOR_POOL[i % len(COLOR_POOL)],
               linestyle=LINESTYLE_POOL[i % len(LINESTYLE_POOL)])
          for i, experiment in enumerate(EXPERIMENTS) if experiment["name"] in results]

plt.style.use("seaborn-v0_8-whitegrid")
FS = {"title": 19, "label": 18, "tick": 16, "legend": 16, "annotation": 14}
fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True)
all_epochs = [e for name in results for e in results[name]["valid_epochs"]]
vmin, vmax = min(all_epochs) * 0.9, max(all_epochs)

for i, basin in enumerate(SCATTER_BASINS):
    first = results[styled[0]["name"]]
    reference = truth_spreads[f"{first['base_climate']}_to_{first['target_climate']}"][basin]
    ax_spread, ax_skill = axes[i, 0], axes[i, 1]
    ax_spread.axhline(y=reference, color="k", linestyle="--", linewidth=2, zorder=5,
                      label=f"CESM2 internal variability ({reference:.3f} {UNITS})")

    notes_spread, notes_skill = [], []
    for experiment in styled:
        data = results[experiment["name"]]
        rmse_vals = np.array(data["rmse"])
        order = np.argsort(rmse_vals)
        rmse_vals = rmse_vals[order]
        spread = np.array(data["metrics"][basin]["spread"])[order]
        skill = np.array(data["metrics"][basin]["correlation"])[order]
        epochs = np.array(data["valid_epochs"])[order]

        for ax, values, notes in [(ax_spread, spread, notes_spread),
                                  (ax_skill, skill, notes_skill)]:
            note = plot_trend(ax, rmse_vals, values, color=experiment["color"],
                              linestyle=experiment["linestyle"])
            if note:
                notes.append(f"{experiment['label']}: {note}")
            scatter = ax.scatter(rmse_vals, values, c=epochs, cmap=cmocean.cm.amp,
                                 marker=experiment["marker"], s=200, alpha=0.8,
                                 edgecolors="k", zorder=10, vmin=vmin, vmax=vmax)

        print(f"\n{experiment['name']} - {basin}")
        for label, values in [("spread", spread), ("skill", skill)]:
            rho, p = stats.spearmanr(rmse_vals, values)
            r, p_pearson = stats.pearsonr(rmse_vals, values)
            print(f"  {label:<7} Spearman rho={rho:+.3f} (p={p:.4f})  "
                  f"Pearson r={r:+.3f} (p={p_pearson:.4f})")

    ax_spread.set_title(f"{panel_letter(2 * i)} Mean-State RMSE vs. Ensemble Spread ({basin})",
                        fontsize=FS["title"], loc="left")
    ax_spread.set_ylabel(f"Ensemble Spread ($N_E{{=}}5$, {UNITS})", fontsize=FS["label"])
    ax_skill.set_title(f"{panel_letter(2 * i + 1)} Mean-State RMSE vs. Response Skill ({basin})",
                       fontsize=FS["title"], loc="left")
    ax_skill.set_ylabel(r"Response Skill ($\rho$ vs. CESM2)", fontsize=FS["label"])

    for ax, notes, (x, y, va) in [(ax_spread, notes_spread, (0.95, 0.75, "center")),
                                  (ax_skill, notes_skill, (0.95, 0.05, "bottom"))]:
        if notes:
            ax.text(x, y, "\n".join(notes), transform=ax.transAxes, fontsize=FS["annotation"],
                    verticalalignment=va, horizontalalignment="right",
                    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

handles = [Line2D([0], [0], marker=e["marker"], color=e["color"], linestyle=e["linestyle"],
                  linewidth=2, label=e["label"], markerfacecolor="gray",
                  markeredgecolor="k", markersize=10) for e in styled]
fig.legend(handles=handles, fontsize=FS["legend"], loc="lower center",
           bbox_to_anchor=(0.45, 0.95), ncol=min(4, len(styled)), framealpha=0.9,
           title="Emulator (marker) / trend (line)", title_fontsize=FS["legend"])
axes[0, 0].legend(fontsize=FS["legend"], loc="center right")
axes[1, 0].legend(fontsize=FS["legend"], loc="center right")

for ax in axes.flat:
    ax.tick_params(axis="both", labelsize=FS["tick"])
    ax.grid(True, linestyle=":", alpha=0.7)
for ax in axes[1, :]:
    ax.set_xlabel(f"Time-Mean Upper-{UPPER_OCEAN_DEPTH}m RMSE vs. CESM2 ({UNITS})",
                  fontsize=FS["label"])

fig.tight_layout(rect=[0, 0, 0.9, 0.95])
cbar = fig.colorbar(scatter, cax=fig.add_axes([0.92, 0.15, 0.02, 0.7]))
cbar.set_label("Epoch Number", fontsize=FS["label"])
cbar.ax.tick_params(labelsize=FS["tick"])
plt.savefig(f"{FIG_DIR}/MultiExperiment_MultiBasin_Epoch_Comparisons.png",
            dpi=300, bbox_inches="tight")
plt.show()
plt.style.use("default")

## Baselines — how much of the response needs internal dynamics

The remaining cells reproduce the supplementary comparisons against the two baselines of
Section S2, both fit on the same piControl data as $\mathcal{F}_{\mathrm{pi}}$ and driven
with the midHolocene forcing:

| model | ocean memory | linear |
|---|---|---|
| CESM2 | — | — |
| Emulator ($\mathcal{F}_{\mathrm{pi}}$, epoch 50) | yes | no |
| LRO — one ridge regression per (x, y, lev) | no | yes |
| Forcing-only network — the emulator's architecture, no state input | no | no |

The two remove different things, so the gap between them is what a nonlinear, non-local
map from the forcing buys, and the gap from the forcing-only network to the emulator is
what ocean memory buys. Skip these cells if the baseline stores have not been generated
(`Scripts/Train_LRO_Updated.py`, `Scripts/Train_Boundary_Forcing_Only.py`).

In [ ]:
MODELS = ["truth", "emulator", "lro", "bfo"]
LABELS = {"truth": "CESM2", "emulator": f"Emulator (epoch {EPOCH_PI})", "lro": "LRO",
          "bfo": "Forcing-only network"}
LABELS_MATH = {"truth": "CESM2", "emulator": r"$\mathcal{F}_{\mathrm{pi}}$", "lro": "LRO",
               "bfo": "Forcing-only"}

for required in (LRO_MEAN, LRO_SEASONAL, LRO_DESEASON, BFO):
    if not os.path.exists(required):
        raise FileNotFoundError(required)

TRUTH_PI = xr.open_zarr(TRUTH["piControl"])
TRUTH_HO = xr.open_zarr(TRUTH["Holocene"])
WET = (TRUTH_PI[VAR] * 0 + 1).isel(time=0, drop=True)
GRID_LON, GRID_LAT = TRUTH_PI["lon"], TRUTH_PI["lat"]

EMU_CTRL = response_path("pi", EPOCH_PI).replace("/yearly_means.zarr", "")
EMU_PERT = response_path("pi", EPOCH_PI, FULL_FORCING).replace("/yearly_means.zarr", "")


def align_fields(fields, exclude=("member",)):
    """Inner-join every model onto one grid so all of them are scored identically.

    The LRO products are y=154 and the emulator / forcing-only products y=180. `member` is
    excluded because the ensemble sizes differ on purpose (10 CESM2 decades, 5 emulator
    initial conditions, 1 for each deterministic baseline).
    """
    keys = [k for k in fields if fields[k] is not None]
    aligned = xr.align(*[fields[k] for k in keys], join="inner", exclude=set(exclude))
    return dict(zip(keys, aligned))


def attach_lonlat(da):
    """Restore the 2-D lon/lat coordinates cartopy needs; the baseline stores drop them."""
    return da.assign_coords(lon=GRID_LON.sel(y=da.y), lat=GRID_LAT.sel(y=da.y))


def masked_area(area, reference):
    return area.where(np.isfinite(reference))


def baseline_timeseries(climate):
    """The four models' monthly state over `climate`, on their shared time window."""
    raw = {
        "truth": (TRUTH_PI if climate == "piControl" else TRUTH_HO)[VAR],
        "emulator": xr.open_zarr(ROLLOUT[("pi", climate)])[VAR],
        "lro": xr.open_zarr(f"{LRO_DESEASON}/timeseries_{climate}.zarr")[VAR],
        "bfo": xr.open_zarr(f"{BFO}/timeseries_{climate}.zarr")[VAR],
    }
    dummies = [xr.DataArray(np.zeros(d.sizes["time"]), dims="time",
                            coords={"time": d.time.values}) for d in raw.values()]
    times = xr.align(*dummies, join="inner")[0].time.values
    return {k: v.sel(time=times) for k, v in raw.items()}


print("LRO (mean)  :", os.path.basename(LRO_MEAN))
print("forcing-only:", os.path.basename(BFO))

### Figure S21 — time-mean response, all four models

The emulator column is the same quantity as Figure 6's middle column. Nothing is
stippled for the two baselines: they are deterministic given the forcing climatologies,
so no spread exists to test against.

In [ ]:
YEAR_WINDOW = slice(-25, None)
sections, spreads, counts, areas, metrics = {}, {}, {}, {}, {}

for basin in BASINS:
    mask = basin_mask(basin)
    lo, hi = basin_bounds(mask)
    sel = dict(lev=slice(0, MAX_DEPTH))

    # Members kept 3-D so every model is scored on the same points before any averaging.
    members = {
        "truth": ((TRUTH_HO[VAR] - TRUTH_PI[VAR]).sel(**sel)
                  .coarsen(time=120, boundary="trim")
                  .construct({"time": ("chunk", "month_in_chunk")})
                  .mean("month_in_chunk").rename({"chunk": "member"})),
        "emulator": ((xr.open_zarr(f"{EMU_PERT}/yearly_means.zarr")[VAR]
                      - xr.open_zarr(f"{EMU_CTRL}/yearly_means.zarr")[VAR]).sel(**sel)
                     .isel(year=YEAR_WINDOW).mean("year").rename({"init_time": "member"})),
    }
    for key, path in [("lro", LRO_MEAN), ("bfo", BFO)]:
        field = xr.open_zarr(f"{path}/yearly_means.zarr")[VAR].sel(**sel)
        if "month" in field.dims:
            field = field.mean("month")
        members[key] = field.expand_dims("member")

    # Drop any coordinate on 'member', or align() would intersect the 10 CESM2 decades
    # against the baselines' single member.
    members = align_fields({k: v.drop_vars("member", errors="ignore").compute()
                            for k, v in members.items()})
    counts[basin] = {k: int(v.sizes["member"]) for k, v in members.items()}

    profiles = {k: zonal_profile(v, mask, WET) for k, v in members.items()}
    sections[basin] = align_fields({k: v.mean("member").sel(y=slice(lo, hi)).compute()
                                    for k, v in profiles.items()})
    spreads[basin] = align_fields({k: v.std("member").sel(y=slice(lo, hi)).compute()
                                   for k, v in profiles.items()})
    truth = sections[basin]["truth"]
    areas[basin] = masked_area(section_area().sel(y=truth.y, lev=truth.lev), truth)
    metrics[basin] = {m: score(sections[basin][m], truth, areas[basin]) for m in MODELS[1:]}
    line = "  ".join(f"{m}: rho={metrics[basin][m]['rho']:.2f} amp={metrics[basin][m]['amp']:.2f}"
                     for m in MODELS[1:])
    print(f"  {basin:<9} {line}")

fig, axes = plt.subplots(len(BASINS), 4, figsize=(21, 12), constrained_layout=True)
for i, basin in enumerate(BASINS):
    out, spread, area = sections[basin], spreads[basin], areas[basin]
    # Scale to CESM2, not to the largest column: one surviving LRO extreme would otherwise
    # set the range and flatten all four panels.
    vmax = float(np.abs(out["truth"]).max())
    norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)

    for j, model in enumerate(MODELS):
        ax = axes[i, j]
        mesh = out[model].plot.contourf(ax=ax, x="y", y="lev", cmap=cmocean.cm.balance,
                                        norm=norm, add_colorbar=False, levels=25,
                                        yincrease=False, extend="both", add_labels=False)
        stipple_section(ax, insignificance_mask(out[model], spread[model],
                                                counts[basin][model]))
        linear_piecewise_scale(500, 2, ax=ax)
        ax.grid(linestyle=":", alpha=0.6)
        head = LABELS[model] if i == 0 else ""
        if j > 0:
            m = metrics[basin][model]
            line = rf"$\rho$={m['rho']:.2f}, RMSE={m['rmse']:.3f}, $\sigma/\sigma_t$={m['amp']:.2f}"
            head = f"{head}\n{line}" if head else line
        ax.set_title(head, fontsize=10 if i else 12)
        ax.set_xlabel("latitude" if i == len(BASINS) - 1 else "")
        if j == 0:
            ax.set_ylabel(f"{basin}\ndepth (m)", fontsize=12)

        for region in [r for r in METRIC_REGIONS if r["basin"] == basin]:
            ax.add_patch(plt.Rectangle(
                (region["lat"][0], region["depth"][0]),
                region["lat"][1] - region["lat"][0],
                region["depth"][1] - region["depth"][0],
                linewidth=2.0, edgecolor=region["color"], linestyle="--",
                facecolor="none", zorder=10, clip_on=True))
            sub = dict(y=slice(*region["lat"]), lev=slice(*region["depth"]))
            if j == 0:
                text = r"$\mathbf{" + region["name"].replace(" ", r"\ ") + r"}$"
            else:
                s = score(out[model].sel(**sub), out["truth"].sel(**sub), area.sel(**sub))
                text = r"$\rho=$" + f"{s['rho']:.2f}\nRMSE={s['rmse']:.2f}"
            lo_x, hi_x = ax.get_xlim()
            deep, shallow = ax.get_ylim()
            rx0, rx1 = np.clip(region["lat"], lo_x, hi_x)
            ry0 = max(region["depth"][0], min(shallow, deep))
            ry1 = min(region["depth"][1], max(shallow, deep))
            ax.text(rx0 + 0.02 * (rx1 - rx0), ry1 - 0.02 * (ry1 - ry0), " " + text,
                    color="black", ha="left", va="bottom", fontsize=7.5, zorder=12,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.2,
                              ec=region["color"], lw=1.5))
    fig.colorbar(mesh, ax=axes[i, :], shrink=0.85, pad=0.01, label=f"{VAR} ({UNITS})")

fig.suptitle("Time-mean mid-Holocene minus piControl response — stippling marks "
             "|response| inside the 95% CI of zero", fontsize=15)
plt.savefig(f"{FIG_DIR}/1_mean_response_sections_{VAR}.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure S20 — change in the seasonal range, all four models

The LRO column here is the `monthly_seasonal` operator, the mode fit on anomalies about
the long-term mean with the seasonal cycle retained, which is the one designed to be the
null for a change in seasonality.

In [ ]:
SEASONAL_MAX_DEPTH = 500
pacific = MASKS["Pacific"]
lo, hi = basin_bounds(pacific)


def seasonal_range_3d(da):
    """Δ(SON − MAM) of a monthly field over its last 600 months."""
    field = da.sel(lev=slice(0, SEASONAL_MAX_DEPTH)).isel(time=slice(-600, None))
    seasons = field.groupby("time.season").mean("time").compute()
    return seasons.sel(season="SON") - seasons.sel(season="MAM")


fields = {
    "truth": seasonal_range_3d(TRUTH_HO[VAR]) - seasonal_range_3d(TRUTH_PI[VAR]),
    "emulator": (seasonal_range_3d(xr.open_zarr(ROLLOUT[("pi", "Holocene")])[VAR])
                 - seasonal_range_3d(xr.open_zarr(ROLLOUT[("pi", "piControl")])[VAR])),
}
for key, path in [("lro", LRO_SEASONAL), ("bfo", BFO)]:
    fields[key] = xr.open_zarr(f"{path}/seasonal_range.zarr")[VAR].sel(lev=slice(0, SEASONAL_MAX_DEPTH))

fields = align_fields({k: v.compute() for k, v in fields.items()})
seasonal = align_fields({k: zonal_profile(v, pacific, WET).sel(y=slice(lo, hi)).compute()
                         for k, v in fields.items()})
area = masked_area(section_area(SEASONAL_MAX_DEPTH).sel(y=seasonal["truth"].y,
                                                        lev=seasonal["truth"].lev),
                   seasonal["truth"])
metrics = {m: score(seasonal[m], seasonal["truth"], area) for m in MODELS[1:]}
for model, m in metrics.items():
    print(f"  {model:<9} rho={m['rho']:.2f}  RMSE={m['rmse']:.3f}  amp={m['amp']:.2f}")

# No shared y axis: linear_piecewise_scale sets a custom scale per axis, and applying it to
# linked axes corrupts the drawn contour paths.
fig, axes = plt.subplots(2, 2, figsize=(11, 8.8), constrained_layout=True)
vmax = 0.8 * float(np.abs(seasonal["truth"]).max())
norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)

for j, model in enumerate(MODELS):
    ax = axes.flat[j]
    mesh = seasonal[model].plot.contourf(ax=ax, x="y", y="lev", cmap=cmocean.cm.balance,
                                         norm=norm, add_colorbar=False, levels=25,
                                         yincrease=False, extend="both", add_labels=False)
    # Drawn only on latitudes sampled at every plotted level; the ragged columns at the
    # northern basin edge otherwise send the contour shooting up the edge of the panel.
    solid = seasonal[model].where(
        seasonal[model].notnull().sum("lev") == seasonal[model].sizes["lev"])
    contour = solid.plot.contour(ax=ax, colors="k", levels=[0.25], yincrease=False,
                                 add_labels=False)
    ax.clabel(contour, inline=True, fontsize=9)
    linear_piecewise_scale(500, 2, ax=ax)
    ax.grid(linestyle=":", alpha=0.6)
    ax.set_title(LABELS[model], fontsize=13)
    ax.set_xlabel("latitude" if j >= 2 else "")
    ax.set_ylabel("depth (m)" if j % 2 == 0 else "")
    if model != "truth":
        m = metrics[model]
        ax.text(0.02, 0.06, rf"$\rho$={m['rho']:.2f}" "\n" + rf"RMSE={m['rmse']:.3f}",
                transform=ax.transAxes, fontsize=9, va="bottom",
                bbox=dict(boxstyle="round", fc="white", alpha=0.75, ec="0.6"))

fig.colorbar(mesh, ax=axes, shrink=0.85, pad=0.01,
             label=rf"$\Delta$(SON$-$MAM) change ({UNITS})")
fig.suptitle("Pacific: mid-Holocene minus piControl change in the seasonal range", fontsize=15)
plt.savefig(f"{FIG_DIR}/3_seasonal_range_{VAR}.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure S22 — change in variability, all four models

Interannual variability: every model is deseasonalised first. The LRO timeseries are
stored as anomalies about their own monthly climatology while the emulator and the
forcing-only network store the absolute state, so comparing raw σ would score a
deseasonalised model against un-deseasonalised ones. Levels are restricted to the LRO's,
which is why the deep band resolves to 250–775 m.

In [ ]:
VAR_BANDS = [(0, 200), (200, 1000)]
VAR_CLIP = 2.25
LRO_LEVELS = xr.open_zarr(f"{LRO_DESEASON}/timeseries_piControl.zarr").lev.values
FIELDS = {climate: baseline_timeseries(climate) for climate in ["piControl", "Holocene"]}


def band_std(climate, model, levels):
    field = FIELDS[climate][model]
    field = field.sel(lev=field.lev[np.isin(field.lev.values, levels)])
    dz_sel = DZ.sel(lev=field.lev).fillna(0.0)
    return deseasonalize(field).std("time").weighted(dz_sel).mean("lev").compute()


for band in VAR_BANDS:
    levels = LRO_LEVELS[(LRO_LEVELS >= band[0]) & (LRO_LEVELS <= band[1])]
    if len(levels) == 0:
        print(f"  {band[0]}-{band[1]} m: no LRO levels, skipped")
        continue
    tag = f"{levels.min():.0f}-{levels.max():.0f}m"
    sigma = {c: {m: band_std(c, m, levels) for m in MODELS} for c in FIELDS}
    change = align_fields({m: sigma["Holocene"][m] - sigma["piControl"][m] for m in MODELS})
    change = {m: attach_lonlat(v) for m, v in change.items()}
    area = masked_area(AREACELLO.sel(y=change["truth"].y), change["truth"])
    metrics = {m: score(change[m], change["truth"], area) for m in MODELS[1:]}
    print(f"  {band[0]}-{band[1]} m -> {tag} ({len(levels)} LRO levels): " +
          "  ".join(f"{m}: rho={metrics[m]['rho']:.2f}" for m in MODELS[1:]))

    fig, axes = plt.subplots(2, 2, figsize=(13, 6.4), subplot_kw={"projection": PROJECTION},
                             constrained_layout=True)
    vmax = float(np.nanpercentile(np.abs(change["truth"].values), 99))
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    for k, model in enumerate(MODELS):
        ax = axes[k // 2, k % 2]
        mesh = change[model].plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
                                             cmap=cmocean.cm.balance, norm=norm,
                                             x="lon", y="lat", add_colorbar=False,
                                             add_labels=False)
        ax.add_feature(cfeature.LAND, zorder=2, edgecolor="black", facecolor="lightgray")
        ax.coastlines(zorder=3, linewidth=0.6)
        gl = ax.gridlines(draw_labels=True, alpha=0.4, linestyle=":")
        gl.top_labels = gl.right_labels = False
        gl.left_labels = k % 2 == 0
        gl.bottom_labels = k // 2 == 1
        ax.set_title(LABELS_MATH[model], fontsize=13)
        if model != "truth":
            m = metrics[model]
            ax.text(0.02, 0.04, rf"$\rho$={m['rho']:.2f}  RMSE={m['rmse']:.3f}",
                    transform=ax.transAxes, fontsize=9,
                    bbox=dict(boxstyle="round", fc="white", alpha=0.75, ec="0.6"))

    fig.colorbar(mesh, ax=axes, shrink=0.8, pad=0.02, label=rf"$\Delta\sigma$ ({UNITS})")
    fig.suptitle(f"Change in interannual variability, {band[0]}–{band[1]} m "
                 f"(mid-Holocene minus piControl)", fontsize=14)
    plt.savefig(f"{FIG_DIR}/6_variability_change_maps_{band[0]}-{band[1]}m_{VAR}.png",
                dpi=200, bbox_inches="tight")
    plt.show()